# Proteasome Challenge 2026

### A guided laboratory in molecular electronic structure

**IBM Quantum**  
*Thaddeus Pellegrini, Quantum Algorithm Engineer*

The 20S proteasome contains a β5 catalytic site targeted by boronic-acid proteasome inhibitors. **MLN2238**, the active boronic-acid form of ixazomib, provides the molecular system investigated in this challenge. [Tanaka (2009)](https://doi.org/10.2183/pjab.85.12); [Lee et al. (2011)](https://doi.org/10.1158/1078-0432.CCR-11-0636).

**The investigation:** Compute the electronic energy of isolated ixazomib through Hartree–Fock, embedded-wavefunction (EWF) fragmentation, classical and quantum-assisted correlation methods, and molecular energy reconstruction.

The workflow partitions the molecule into embedded fragments and assigns correlation solvers according to their orbital spaces. Smaller fragments are treated using **full configuration interaction (FCI)**, intermediate fragments using **coupled-cluster singles and doubles (CCSD)**, and selected larger fragments using **TrimSQD**, a quantum-assisted configuration-recovery method.

Participants will determine the fragment distribution, investigate the computational tradeoffs, and reconstruct the molecular electronic energy from the correlated fragment results.

**Scientific scope:** This experiment investigates molecular electronic energy, not protein–ligand binding energy or inhibitor potency.

## Getting started

The Proteasome Challenge is a guided molecular electronic-structure laboratory, progressing from Hartree–Fock through embedded-wavefunction (EWF) fragmentation, quantum-assisted correlation calculations, and molecular energy reconstruction.

The workflow comprises seven scientific sections and **six graded exercises worth 100 points**.

### Challenge roadmap & grading

| Section | Objective | Exercise | Points |
|---|---|---|---:|
| 0 · Setup | Configure resources and initialize the persistent run | — | — |
| 1 · Hartree–Fock | Establish the molecular mean-field reference | 1 | 10 |
| 2 · EWF fragmentation | Construct and validate embedded fragments | 2 | 12 |
| 3 · Solver allocation | Configure classical and quantum fragment solvers | — | — |
| 4 · Quantum selection | Identify quantum-treated fragments | — | — |
| 5 · SQD laboratory | Benchmark CCSD, SQD, and TrimSQD; optimize solver settings | 3 & 4 | 15 + 20 |
| 6 · Hybrid campaign | Complete the FCI, CCSD, and TrimSQD fragment calculations | 5 | 33 |
| 7 · Reconstruction | Assemble and validate the molecular electronic energy | 6 | 10 |
| | **Total** | **6 exercises** | **100** |

### Grading policy

- **All-or-nothing:** Each exercise awards its full point value only when every acceptance criterion is satisfied. No partial credit is awarded.
- **Numerical accuracy:** Results must satisfy the reference values and tolerances specified for each exercise.
- **Resubmissions:** Unsuccessful exercises may be corrected and resubmitted. Each exercise contributes points only once.
- **Final score:** The sum of the six exercise scores, up to 100 points.

**Experimental integrity and winner verification**

Artifacts from each experimental stage must be retained and made available for organizer inspection. A passing automated grader score does not constitute final verification.

Organizers reserve the right to invalidate scores that cannot be substantiated by authentic experimental artifacts. Evidence of grader manipulation, fabricated results, or other challenge-integrity violations may result in disqualification at the organizers' sole discretion, without appeal.

### Execution conventions

The notebook maintains a persistent `RUN_ID`, task registry, and scientific artifacts. Existing runs can be resumed without repeating completed calculations.

- **Execution controls:** Stage flags authorize scientific operations, including potentially paid Serverless and QPU jobs. All submission and advancement controls are disabled by default and must be enabled deliberately.
- **Campaign lifecycle:** Launch and advancement are separate operations. Follow the Section 6 operator's manual for submission, monitoring, reconciliation, and recovery.
- **Result states:** `PENDING` does not imply successful completion. Verify recorded results and provenance before proceeding.
- **Artifact preservation:** Retain the current run's scientific records for final organizer verification.

### Environment setup

From `proteasome-inhibitor/`, create the credentials file:

```bash
cp .env.example .env
```

Configure the IBM Quantum and QRMI credentials and backend settings described in `.env.example`.

The `.env` file contains sensitive credentials, is excluded from version control, and must not be committed.

Proceed through the control panel, environment initialization, and preflight checks before executing the scientific workflow.

<div class="alert alert-block alert-success">

✏️ **Your task:** Complete the cell below to stablish your team's connection to the QDC 2026 challenge grading system.

</div>

In [ ]:
from qc_grader.challenges.qdc_2026 import join_team

TEAM_NAME = ""  # YOUR TEAM NAME, the same for every member

if TEAM_NAME:
    join_team(TEAM_NAME)
else:
    print("Set TEAM_NAME to your team's name and run this cell again: answers are only graded for a team.")

## Control Panel

### 0A · Configure campaign execution

Define the execution controls for the full hybrid campaign and independent single-fragment experiments.

<div class="alert alert-block alert-success">

✏️ **Your task:** Review the execution flags before running the notebook.

</div>

- `USE_FULL_LIVE_TRIAL` — Selects the full hybrid campaign workflow. This setting alone does not authorize execution.
- `ARM_FULL_LIVE_TRIAL` — Authorizes campaign execution when explicitly enabled.
- `LAUNCH_FULL_TRIAL` — Enables initial campaign submission.
- `ADVANCE_FULL_TRIAL` — Enables subsequent reconciliation and advancement.
- `LIVE_QPU_SAMPLING` — Selects live QPU acquisition or measurement replay in the independent single-fragment laboratory.

**Initial execution:** Campaign arming, launch, and advancement are all disabled by default. `LAUNCH_FULL_TRIAL` and `ADVANCE_FULL_TRIAL` must never be enabled simultaneously.

**Important:** `LIVE_QPU_SAMPLING` controls the independent fragment laboratory, not the full hybrid campaign. Full-campaign QPU execution is governed by its own authorization and launch workflow.

The remaining controls govern polling, artifact verification, data refresh, and repeat sampling.

**Expected outcome:** Validated execution controls with no conflicting launch and advancement settings.

This cell **defines settings only** and performs no remote operations. Execution must be explicitly authorized at the appropriate stage. Review the Section 6 operating instructions before launching computational work.

In [ ]:

# ── 0A · Campaign controls ────────────────────────────────────────

# Select the full experimental workflow.
# Selection does not authorize execution.
USE_FULL_LIVE_TRIAL = True

# Campaign execution — initially disarmed.
ARM_FULL_LIVE_TRIAL = False
LAUNCH_FULL_TRIAL = False
ADVANCE_FULL_TRIAL = False

# General settings.
POLL_SECONDS = 30
ALLOW_UNVERIFIED_BUNDLE = False
ALLOW_CHALLENGE_DATA_REFRESH = False

# Independent single-fragment experiments.
LIVE_QPU_SAMPLING = False
ALLOW_SECOND_QPU_SAMPLING = False

if LAUNCH_FULL_TRIAL and ADVANCE_FULL_TRIAL:
    raise ValueError(
        "Launch and advancement cannot be enabled simultaneously."
    )


### 0B · Configure campaign resources

Define the QPU measurement budget and computational resources for the distributed fragment campaign.

<div class="alert alert-block alert-success">

✏️ **Your task:** Choose `FULL_TRIAL_SHOTS`, the number of measurement shots acquired per quantum fragment. Consider the statistical quality of the measurements and the available QPU execution budget.

</div>

Inspect the remaining settings governing Serverless concurrency, worker resources, QPU submission, and monitoring.

The fragment allocation will be determined from the computed fragment catalog later in the notebook.

**Expected outcome:** A valid execution profile ready for computational resource validation.

This cell defines configuration only and submits no calculations.

In [ ]:

# ── 0B · Campaign resources ───────────────────────────────────────

# Choose the QPU measurement budget per quantum fragment.
FULL_TRIAL_SHOTS = FILL_IN_QPU_SHOTS

FULL_LIVE_TRIAL = {
    "classical_max_concurrent": None,
    "classical_resource_family": "l",
    "postprocess_max_concurrent": None,
    "postprocess_resource_family": "l",
    "serverless_max_total_jobs": 42,
    "postprocess_reserved_slots": 1,
    "qpu_submit_max_per_call": None,
    "qpu_submit_time_box_seconds": 1500.0,
    "max_status_queries": 100,
}


### 0C · Configure compute backends

Select the computational backends for the molecular simulation workflow.

<div class="alert alert-block alert-success">

✏️ **Your task:** Inspect `COMPUTE_CONFIG`, which specifies:

- `meanfield_backend` — Hartree–Fock calculations.
- `classical_backend` — Classical fragment solvers.
- `quantum_postprocess_backend` — SQD/TrimSQD classical post-processing.

</div>

**Expected outcome:** A backend configuration ready for validation in **1D**.

This cell **defines settings only** and performs no calculations or remote operations.

In [ ]:
# ── 0C · Compute backends ─────────────────────────────────────────

COMPUTE_CONFIG = {
    "meanfield_backend": "cpu",
    "classical_backend": "cpu",
    "quantum_postprocess_backend": "cpu",
}

### 1A · Initialize scientific libraries and challenge tools

Load the scientific libraries, challenge interfaces, computational configurations, and workflow utilities required throughout the laboratory.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to initialize the molecular simulation tools, execution helpers, solver configurations, and plotting style.

</div>

**Expected outcome:** Successful imports with no errors.

This cell **performs no scientific calculations or remote operations**. The following cell imports the six official Proteasome Challenge grading functions.

In [ ]:
# ── 1A · Scientific libraries and challenge tools ───────────────────

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from proteasome_challenge import (
    Challenge, ChallengeError, LIGANDS, BASIS,
    FCI_THRESHOLD, SQD_THRESHOLD, SELECTED_LIGAND,
    resolve_qpu_backend,
)
from proteasome_challenge._plot_style import (
    apply as _apply_style, BLUE, ORANGE, GRAY, ligand_color,
)
from proteasome_challenge import full_trial as ft
from proteasome_challenge.hf_stage import MoleculeSpec, HFConfig
from proteasome_challenge.ewf_stage import (
    FragmentationConfig, BathConfig, EWFConfig, Stage2Spec,
    ewf_catalog_from_result,
)
from proteasome_challenge.run_control_plane import (
    validate_compute_config, init_run, load_stage_ref, publish_stage2,
    collect_manifest_inputs, collect_manifest_inputs_generic,
    freeze_run_manifest, ensure_fragment_tasks, inspect_run_status,
)
from proteasome_challenge.solver_assignments import (
    AllocationRule, SolverAllocationPolicy, SolverContract,
    list_registered_solvers, assign_solvers_by_size,
)
from proteasome_challenge.solver_config import (
    FCIConfig, CCSDConfig, SQDConfig, TrimSQDConfig, Stage3Config,
)

_apply_style()

In [ ]:
from qc_grader.challenges.qdc_2026.lab_proteasome import (
    grade_lab_proteasome_ex1,
    grade_lab_proteasome_ex2,
    grade_lab_proteasome_ex3,
    grade_lab_proteasome_ex4,
    grade_lab_proteasome_ex5,
    grade_lab_proteasome_ex6,
)

### 1B · Configure the quantum backend

Load the IBM Quantum backend configuration from the local `.env` file.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to initialize `QPU_BACKEND` from `IBM_QUANTUM_BACKEND`.

</div>

**Expected outcome:** The configured backend name, or `NOT SET` if no backend is specified.

This cell **performs no remote operations** and does not verify hardware availability. Backend access is checked during subsequent execution readiness checks.

In [ ]:
# ── 1B · Quantum backend ─────────────────────────────────────────────

import os as _os
from dotenv import find_dotenv, load_dotenv

_env_path = find_dotenv(usecwd=True)
if _env_path:
    load_dotenv(_env_path, override=True)

QPU_BACKEND = _os.environ.get("IBM_QUANTUM_BACKEND", "").strip() or None

print(f"QPU backend: {QPU_BACKEND or 'NOT SET'}")

### Challenge artifacts

The challenge maintains a persistent record of each experiment, allowing calculations to be resumed and results independently verified.

| Artifact | Purpose |
|---|---|
| Scientific inputs | Molecular geometry, computational settings, and reference data supplied with the challenge |
| Stage 1–2 results | Hartree–Fock results, embedded fragment Hamiltonians, and orbital information |
| RunManifest | Frozen scientific configuration, fragment identities, and solver assignments |
| Execution records | Task states, QPU measurements, Serverless jobs, fragment results, and provenance |
| Stage 4 results | Reconstructed molecular energy and fragment correlation contributions |
| Organizer report | Consolidated experimental audit exported as HTML, JSON, and CSV |

<div class="alert alert-block alert-success">

✏️ **Your task:** Preserve the artifacts generated throughout the workflow. All graded results must be traceable to authentic experimental records associated with the current `RUN_ID`.

</div>

The final organizer report consolidates these records for **independent verification of grader results and winning submissions**.

Historical results and supplied reference data must not be presented as newly computed results.

### 1C · Initialize the challenge

Initialize the challenge interface and inspect the configured scientific environment.

<div class="alert alert-block alert-success">

✏️ **Your task:** 
Run the cell to create `lab` using `Challenge.from_environment()` and display its status.

</div>

**Expected outcome:** An initialized challenge interface and an environment status report.

This cell **does not execute scientific calculations or submit remote jobs**. Proceed to **1D** to configure computational resources.

In [ ]:
# ── 1C · Initialize the challenge ────────────────────────────────────

lab = Challenge.from_environment(
    allow_unverified_bundle=ALLOW_UNVERIFIED_BUNDLE
)

lab.show_status()

### 1D · Configure computational resources

Validate the computational settings and establish the artifact-storage location for the distributed experiment.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to validate `COMPUTE_CONFIG`, initialize `FULL_TRIAL_PROFILE`, and inspect the Serverless concurrency limit.

</div>

**Expected outcome:** The configured `ARTIFACT_ROOT` and printed Serverless concurrency ceiling.

This cell **does not deploy software or execute calculations**. Proceed to **1E** to establish the persistent campaign identity.

In [ ]:
# ── 1D · Computational resources ─────────────────────────────────────

COMPUTE_CONFIG = validate_compute_config(COMPUTE_CONFIG)

ARTIFACT_ROOT = lab.ckpt_dir / "phase5"

FULL_TRIAL_PROFILE = ft.FullTrialProfile(**FULL_LIVE_TRIAL)

ft.assert_legacy_flags_off(globals())

print(
    "Serverless concurrency ceiling:",
    FULL_TRIAL_PROFILE.serverless_max_total_jobs,
)

### 1E · Establish campaign identity

Assign a persistent experiment identifier and isolate scientific artifacts for the current campaign.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell with `USE_FULL_LIVE_TRIAL = True` to establish `RUN_ID` and scope Stage 1–2 artifacts to the experiment.

</div>

**Expected outcome:** The selected `RUN_ID`, artifact scope, and structural execution guidance.

The identifier persists across notebook sessions, allowing the same experiment to be resumed without changing its identity.

This cell **does not submit remote calculations**. Proceed to **2** to deploy the Serverless bundle.

In [ ]:
# ── 1E · Establish campaign identity ─────────────────────────────────

if USE_FULL_LIVE_TRIAL:
    RUN_ID = ft.stable_run_id(
        lab.ckpt_dir.parent / "full_live_trial_run_id.json"
    )

    print(f"FULL_LIVE_TRIAL selected. RUN_ID = {RUN_ID}")

    _scope = ft.scope_trial_data(
        lab, ARTIFACT_ROOT, RUN_ID
    )

    print(f"Stage 1/2 outputs scoped to {_scope['scope']}")
    print(ft.STRUCTURAL_NOTE)

### 2 · Deploy the Serverless bundle

Deploy the computational software required for remote Hartree–Fock, EWF fragmentation, and correlated fragment calculations.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell with `RUN_DEPLOY = True` to deploy the Serverless bundle.

</div>

**Expected outcome:** Successful deployment and a printed bundle fingerprint identifying the local software contents.

Deployment is a **remote infrastructure operation**, but does not itself execute scientific calculations or sample the QPU.

Keep the deployed software consistent throughout the experiment. Redeploy if its contents change.

In [ ]:

# ── 2 · Deploy the Serverless bundle ──────────────────────────────

RUN_DEPLOY = False

if RUN_DEPLOY:
    lab.deploy_bundle()

print("Bundle fingerprint:", lab.bundle_fingerprint())


### 3 · Initial preflight

Check the scientific reference data and quantum execution configuration before beginning the experiment.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to inspect:

- **Reference data:** Confirm the required benchmark file exists.
- **IBM Quantum credentials:** Check for a configured authentication token.
- **QPU backend:** Inspect the selected backend.

</div>

**Expected outcome:** `Initial preflight: PASSED`, with the QPU configuration status displayed.

This is a **local configuration check**, not a remote authentication or hardware-readiness test. A passing result confirms reference-data availability; quantum credentials and backend accessibility require subsequent verification.

The cell is **read-only** and performs no calculations.

In [ ]:
# ── 3 · Initial preflight ─────────────────────────────────────────

import os

# Scientific reference data.
checks_passed = lab.benchmark_path.is_file()

print(
    "Reference data:",
    "OK" if checks_passed else "MISSING"
)

# Quantum execution configuration.
token_present = bool(
    os.environ.get("IBM_QUANTUM_TOKEN")
    or os.environ.get("QISKIT_IBM_TOKEN")
)

backend_present = bool(QPU_BACKEND)

print(
    "IBM Quantum credentials:",
    "Detected" if token_present else "Not found in environment"
)
print("QPU backend:", QPU_BACKEND or "Not configured")

print(
    "\nInitial preflight:",
    "PASSED" if checks_passed else "INCOMPLETE"
)

print(
    "QPU configuration:",
    "Present (not yet verified)" if token_present and backend_present
    else "Requires verification before quantum execution"
)

### 0.3 · Initialize the persistent run

Each experiment uses a persistent `RUN_ID` linking its task registry, scientific artifacts, and frozen `RunManifest`.

<div class="alert alert-block alert-success">

✏️ **Your task:** Execute the following two cells:

1. **4 · Verify campaign readiness:** Check the current run's clean-start conditions.
2. **5 · Initialize or resume the campaign:** Open `run_ctx` and inspect its run identity and manifest status.

</div>

**Expected outcome:** A verified run identity and a manifest status of `not yet frozen` until Stage E.8.

Preserve the same `RUN_ID` throughout the experiment. These cells **do not submit QPU or Serverless calculations**.

In [ ]:

# ── 4 · Verify campaign readiness ────────────────────────────────

RUN_INIT_RUN = False

run_ctx = globals().get("run_ctx")

if USE_FULL_LIVE_TRIAL:
    _clean_start = ft.ensure_clean_start(
        lab, RUN_ID, enforce=RUN_INIT_RUN
    )
    print(ft.format_clean_start(_clean_start))


In [ ]:

# ── 5 · Initialize or resume the campaign ─────────────────────────

if run_ctx is not None:
    run_ctx.close()

run_ctx = init_run(
    ARTIFACT_ROOT,
    RUN_ID,
    compute_config=COMPUTE_CONFIG,
)

print(f"Run {run_ctx.run_id}: {'created' if run_ctx.created else 'reopened'}")
print(
    f"Manifest: {len(run_ctx.manifest.fragment_ids)} fragments"
    if run_ctx.manifest else "Manifest: not yet frozen"
)

## Exercise 1

## 1 · Molecular specification and Hartree–Fock

Hartree–Fock (HF) approximates the electronic wavefunction using an optimized single Slater determinant. In a finite orbital basis, the Roothaan–Hall equations are

$$
\mathbf{F}\mathbf{C}
=
\mathbf{S}\mathbf{C}\boldsymbol{\varepsilon},
$$

where $\mathbf{F}$ is the Fock matrix, $\mathbf{S}$ is the orbital-overlap matrix, and $\mathbf{C}$ contains the molecular-orbital coefficients.

**Reference:** [Roothaan (1951), Eq. (58′)](https://doi.org/10.1103/RevModPhys.23.69).

### Exercise 1 · Establish the Hartree–Fock reference

**Objective:** Compute the molecular Hartree–Fock reference energy for ixazomib.

**01 · Configure the molecular system**

Inspect `MOLECULE` and `HF_CONFIG`:

| Configuration | Parameters |
|---|---|
| Molecular specification | `path`, `units`, `charge`, `spin` |
| Hartree–Fock | `method`, `basis` |

Consult *Molecular Quantum Computations on a Protein* for the scientific specifications.

<div class="alert alert-block alert-success">

✏️ **Your task:** Verify the molecular inputs and mean-field configuration before execution.

</div>

**02 · Execute Hartree–Fock**

With the persistent run initialized and `USE_FULL_LIVE_TRIAL = True`, execute **7A–7C**.

The Stage 1 execution cell uses `RUN_STAGE1 = True` to authorize and obtain the Hartree–Fock result.

Inspect `hf_status` and `mf_ref` to confirm that the calculation completed successfully and produced a valid mean-field reference.

**03 · Submit the energy**

The Exercise 1 submission cell requires you to report the computed Hartree–Fock energy in Hartree. Enter your answer, enable submission, and call `grade_lab_proteasome_ex1(bundle)`.

**Grading · Exercise 1**

| Criterion | Requirement |
|---|---|
| Molecular system | Ixazomib |
| Submitted quantity | Hartree–Fock total energy |
| Energy agreement | ±0.001 Ha |

**If grading fails:** Verify the geometry, coordinate units, charge, spin, method, and basis before recomputing and resubmitting.

**Outcome:** A validated Hartree–Fock reference for the subsequent EWF fragmentation.

### 6 · Define the ixazomib molecule

Specify the molecular geometry and electronic state used throughout the challenge.

<div class="alert alert-block alert-success">

✏️ **Your task:** Inspect the supplied ixazomib geometry and complete `MOLECULE` by determining:

- **Coordinate units** — The length units used in the geometry file.
- **Molecular charge** — The net electrical charge of the molecular system.
- **Spin** — The difference between the numbers of alpha and beta electrons.

</div>

Use the `FILL_IN_*` placeholders to supply your answers.

**Expected outcome:** A complete molecular specification displaying the molecular name, geometry path, units, charge, and spin.

This cell **defines scientific inputs only** and performs no calculations. Proceed to **7A** to configure Hartree–Fock.

In [ ]:

# ── 6 · Define the ixazomib molecule ─────────────────────────────

_molecule_path = lab.bundle_dir / "configs" / "system_ixazomib.xyz"

if not _molecule_path.is_file():
    raise FileNotFoundError(f"Geometry not found: {_molecule_path}")

MOLECULE = MoleculeSpec(
    name="ixazomib",
    path=str(_molecule_path),
    units=FILL_IN_UNITS,
    charge=FILL_IN_CHARGE,
    spin=FILL_IN_SPIN,
)

print(f"Molecule: {MOLECULE.name}")
print(f"Geometry: {MOLECULE.path}")
print(f"Units: {MOLECULE.units} | Charge: {MOLECULE.charge} | Spin: {MOLECULE.spin}")


### 7A · Configure Hartree–Fock

Choose the mean-field method and orbital basis for the molecular calculation.

<div class="alert alert-block alert-success">

✏️ **Your task:** Complete `HF_CONFIG` by determining:

- **Method:** The appropriate Hartree–Fock method for the molecular electronic state.
- **Basis:** The orbital basis set used throughout the challenge's computational workflow.

</div>

Replace the `FILL_IN_*` placeholders with your selections.

**Expected outcome:** A valid mean-field configuration displaying the selected method and basis set.

This cell **defines scientific settings only** and performs no calculations. Proceed to **7B** to execute Hartree–Fock.

In [ ]:

# ── 7A · Hartree–Fock configuration ───────────────────────────────

HF_CONFIG = HFConfig(
    method=FILL_IN_HF_METHOD,
    basis=FILL_IN_BASIS_SET,
)

print(f"Mean-field method: {HF_CONFIG.method}")
print(f"Basis set: {HF_CONFIG.basis}")


### 7B · Execute Hartree–Fock

Establish the molecular Hartree–Fock reference using your selected geometry, electronic state, method, and basis.

<div class="alert alert-block alert-success">

✏️ **Your task:** Verify your scientific inputs and initialize the persistent campaign. When ready, set `RUN_STAGE1 = True` to execute the calculation.

</div>

The cell authorizes Stage 1 and uses `lab.ensure_hf()` to retrieve an existing result or submit a new calculation.

**Expected outcome:** A completed Hartree–Fock calculation associated with the current `RUN_ID`.

**Execution is disabled by default.** Enabling this stage may consume remote Serverless resources but does not use the QPU.

Proceed to **7C** to inspect the result.

In [ ]:

# ── 7B · Execute Hartree–Fock ─────────────────────────────────────

RUN_STAGE1 = False

if RUN_STAGE1:
    if not USE_FULL_LIVE_TRIAL or run_ctx is None:
        raise RuntimeError("Initialize the full campaign before Stage 1.")

    ft.authorize_stage_execution(
        "stage1",
        run_ctx,
        allow_data_refresh=ALLOW_CHALLENGE_DATA_REFRESH,
        lab=lab,
    )

    hf_result = lab.ensure_hf(
        MOLECULE,
        HF_CONFIG,
        run_ctx=run_ctx,
        allow_submit=True,
        poll_seconds=POLL_SECONDS,
    )


### 7C · Inspect Hartree–Fock results

Inspect the completed Hartree–Fock calculation and retrieve its persistent mean-field reference.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to examine `hf_status` and load `mf_ref` from the current run.

</div>

Confirm that the calculation completed successfully and that the mean-field reference is available.

**Expected outcome:** A successful Hartree–Fock status summary and a valid `mf_ref`.

This cell is **read-only** and submits no calculations. The reference is required for Stage 2 EWF fragmentation.

In [ ]:
# ── 7C · Inspect Hartree–Fock results ─────────────────────────────

hf_status = lab.hf_status(
    MOLECULE,
    HF_CONFIG,
    run_ctx=run_ctx,
)

print(hf_status.summary())

mf_ref = load_stage_ref(run_ctx, "meanfield")
print(f"Mean-field reference: {mf_ref}")

### Exercise 1 · Submit the Hartree–Fock energy

Report the total Hartree–Fock energy obtained from your molecular calculation.

<div class="alert alert-block alert-success">

✏️ **Your task:** Inspect the verified Stage 1 result, determine the energy in Hartree, and replace `FILL_IN_HF_TOTAL_ENERGY` with your numerical answer.

</div>

Set `SUBMIT_EXERCISE_1 = True` when ready to submit.

**Expected outcome:** Grader feedback for your computed Hartree–Fock energy.

Grading submission is disabled by default.

In [ ]:

# ── Exercise 1 · Submit Hartree–Fock energy ───────────────────────

SUBMIT_EXERCISE_1 = False

if SUBMIT_EXERCISE_1:
    if mf_ref is None or not hf_status.ok:
        raise RuntimeError(
            "Complete and verify Stage 1 before submitting Exercise 1."
        )

    bundle = {
        "hartree_fock_energy": float(FILL_IN_HF_TOTAL_ENERGY),
    }

    grade_lab_proteasome_ex1(bundle)


## Exercise 2

## 2 · Construct embedded fragments

**Embedded-wavefunction (EWF)** calculations partition a molecular system into local fragment-and-bath orbital spaces. The Hartree–Fock reference defines a Schmidt/DMET bath, while MP2 bath-natural orbitals (BNOs) expand the bath using approximate correlation information.

**Reference:** [Nusspickel and Booth (2022), §II.A–B](https://doi.org/10.1103/PhysRevX.12.011046).

### Exercise 2 · Construct and validate EWF fragments

**Objective:** Construct the embedded fragment Hamiltonians for ixazomib and verify their orbital spaces.

**01 · Configure fragmentation**

Inspect `FRAGMENTATION` and `BATH`:

| Configuration | Parameters |
|---|---|
| Fragmentation | `scheme`, `basis`, `iao_minao` |
| Bath construction | `method`, `truncation`, `truncation_kind` |

Consult *Molecular Quantum Computations on a Protein* for the underlying methodology.

The canonical configuration uses atomic fragmentation in an intrinsic atomic orbital (IAO) basis and an MP2 bath with occupation-based truncation.

**02 · Execute EWF fragmentation**

Run **8A–8C** to construct and verify the embedded fragments.

<div class="alert alert-block alert-success">

✏️ **Your task:** Confirm that the Hartree–Fock reference is valid, execute Stage 2 with `RUN_STAGE2 = True`, and inspect the resulting EWF status and run-scoped artifacts.

</div>

Then use **9A–9C** to verify fragment provenance and examine the distribution of spatial orbital counts.

**Expected outcome:** 42 embedded fragments with verified orbital spaces, ready for correlated calculations.

**03 · Inspect and submit the fragments**

After selecting the quantum-eligible fragments in Section 4, inspect `_sel_df`:

- `fragment_id` — fragment identifier
- `n_orb` — spatial orbitals
- `n_occ` — occupied orbitals
- `n_vir` — virtual orbitals

<div class="alert alert-block alert-success">

✏️ **Your task:** Verify the selected fragments and execute the Exercise 2 submission cell:

`grade_lab_proteasome_ex2(bundle)`

</div>

**Grading · Exercise 2**

| Criterion | Requirement |
|---|---|
| Selected fragments | Exactly 11 |
| Fragment identifiers | Exact reference match |
| Orbital counts | Exact `n_orb`, `n_occ`, `n_vir` values |

**If grading fails:** Review the fragmentation and bath configuration, verify the current-run Stage 2 artifacts, and repeat fragment selection before resubmitting.

**Outcome:** A validated collection of embedded fragment Hamiltonians for the subsequent classical and quantum calculations.

## 2 · Construct embedded fragments

**Embedded-wavefunction (EWF)** calculations partition a molecular system into local fragment-and-bath orbital spaces. The Hartree–Fock reference provides the starting point for constructing embedded Hamiltonians, while the bath construction determines which additional orbitals are retained.

**Reference:** [Nusspickel and Booth (2022), §II.A–B](https://doi.org/10.1103/PhysRevX.12.011046).

### Exercise 2 · Construct and validate EWF fragments

**Objective:** Construct the embedded fragment Hamiltonians for ixazomib and verify their orbital spaces.

**01 · Configure fragmentation**

Complete `FRAGMENTATION` and `BATH` by determining the appropriate scientific parameters:

| Configuration | Parameters |
|---|---|
| Fragmentation | `scheme`, `basis`, `iao_minao` |
| Bath construction | `method`, `truncation`, `truncation_kind` |

Consult *Molecular Quantum Computations on a Protein* and the EWF methodology described above to determine the appropriate fragmentation and bath-construction settings.

**Important:** For reproducibility, this challenge requires a bath truncation threshold of **`1e-4`**. Other thresholds may be scientifically valid, but they change the embedded orbital spaces and can invalidate downstream comparisons.

Replace the `FILL_IN_*` placeholders with your selected values.

**02 · Execute EWF fragmentation**

Run **8A–8C** to construct and verify the embedded fragments.

<div class="alert alert-block alert-success">

✏️ **Your task:** Confirm that the Hartree–Fock reference is valid. When ready, set `RUN_STAGE2 = True`, execute Stage 2, and inspect the resulting EWF status and run-scoped artifacts.

</div>

Then use **9A–9C** to verify fragment provenance and examine the distribution of spatial orbital counts.

**Expected outcome:** A verified collection of embedded fragments and their associated orbital spaces, ready for correlated calculations.

**03 · Inspect and submit the fragments**

After selecting the quantum-eligible fragments in Section 4, inspect `_sel_df`:

- `fragment_id` — Fragment identifier
- `n_orb` — Spatial orbitals
- `n_occ` — Occupied orbitals
- `n_vir` — Virtual orbitals

<div class="alert alert-block alert-success">

✏️ **Your task:** Verify the selected fragments, complete the Exercise 2 submission bundle, and submit your results using:

`grade_lab_proteasome_ex2(bundle)`

</div>

**Grading · Exercise 2**

| Criterion | Requirement |
|---|---|
| Selected fragments | Correct selection under the challenge's solver-allocation criteria |
| Fragment identifiers | Exact reference match |
| Orbital counts | Exact `n_orb`, `n_occ`, `n_vir` values |

**If grading fails:** Review the fragmentation and bath configuration, verify the current-run Stage 2 artifacts, and repeat fragment selection before resubmitting.

**Outcome:** A validated collection of embedded fragment Hamiltonians for subsequent classical and quantum calculations.

In [ ]:

# ── 2 · EWF fragmentation and bath configuration ────────────────

FRAGMENTATION = FragmentationConfig(
    scheme=FILL_IN_FRAGMENTATION_SCHEME,
    basis=FILL_IN_FRAGMENTATION_BASIS,
    iao_minao=FILL_IN_IAO_MINAO,
)

# For this challenge, use a bath truncation threshold of 1e-4.
# Other thresholds are scientifically valid, but would alter the
# embedded Hamiltonians and invalidate downstream comparisons.
BATH = BathConfig(
    method=FILL_IN_BATH_METHOD,
    truncation=FILL_IN_BATH_TRUNCATION,
    truncation_kind=FILL_IN_TRUNCATION_KIND,
)

EWF_CONFIG = EWFConfig(
    fragmentation=FRAGMENTATION,
    bath=BATH,
)


### 8A · Execute EWF fragmentation

Construct the embedded fragment-and-bath orbital spaces using the verified Hartree–Fock reference and your selected EWF configuration.

<div class="alert alert-block alert-success">

✏️ **Your task:** Review your scientific inputs and confirm that Stage 1 completed successfully. When ready, set `RUN_STAGE2 = True` to execute or recover the Stage 2 fragmentation calculation.

</div>

The cell verifies the current-run Hartree–Fock reference, authorizes Stage 2 execution, and retrieves or generates the EWF results.

**Expected outcome:** A completed Stage 2 calculation with its fragment artifacts recorded under the current `RUN_ID`.

**Execution is disabled by default.** Enabling this stage may submit remote Serverless calculations but performs no QPU sampling.

Proceed to **8B** to inspect the EWF execution status.

In [ ]:

# ── 8A · Execute EWF fragmentation ───────────────────────────────

RUN_STAGE2 = False

if RUN_STAGE2:
    if not USE_FULL_LIVE_TRIAL or run_ctx is None:
        raise RuntimeError("Initialize the full campaign before Stage 2.")

    # Require a verified Hartree–Fock reference.
    _stage1_ref = load_stage_ref(run_ctx, "meanfield")
    _hf_status = lab.hf_status(
        MOLECULE, HF_CONFIG, run_ctx=run_ctx
    )

    if _stage1_ref is None or _stage1_ref.run_id != RUN_ID:
        raise RuntimeError("Current-run Hartree–Fock reference is missing.")

    if not _hf_status.ok:
        raise RuntimeError("Hartree–Fock reference failed verification.")

    ft.authorize_stage_execution(
        "stage2",
        run_ctx,
        allow_data_refresh=ALLOW_CHALLENGE_DATA_REFRESH,
        lab=lab,
    )

    lab.ensure_ewf(
        MOLECULE,
        HF_CONFIG,
        EWF_CONFIG,
        run_ctx=run_ctx,
        allow_submit=True,
        poll_seconds=POLL_SECONDS,
    )


### 8B · Inspect EWF results

Inspect the execution status of Stage 2 embedded-wavefunction fragmentation.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to retrieve `ewf_status` and examine its summary.

</div>

**Expected outcome:** A successful EWF status confirming that fragmentation completed.

This cell is **read-only**, performs no remote calculations, and can be rerun safely.

Proceed to **8C** to verify the current-run fragment artifact and recover its scientific data.

In [ ]:
# ── 8B · Inspect EWF results ──────────────────────────────────────

ewf_status = lab.ewf_status(
    MOLECULE,
    HF_CONFIG,
    EWF_CONFIG,
    run_ctx=run_ctx,
)

print(ewf_status.summary())

### 8C · Verify EWF fragment data

Verify the completed fragmentation calculation and recover its scientific results.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell after Stage 2 completes. Confirm that the EWF calculation succeeded and that its published artifact belongs to the current `RUN_ID`.

</div>

The cell reconstructs `ewf_stage2_result` from the verified EWF status for subsequent fragment analysis.

**Expected outcome:** `Stage 2 verified`, followed by the authoritative artifact name.

This cell is **read-only** and submits no remote calculations. It can be rerun after Stage 2 completes.

Proceed to **9A** for additional provenance verification.

In [ ]:

# ── 8C · Verify EWF fragment data ────────────────────────────────

if run_ctx is None or run_ctx.run_id != RUN_ID:
    raise RuntimeError("Initialize the current campaign first.")

ewf_ref = load_stage_ref(run_ctx, "embedding_ewf")

if not ewf_status.ok:
    raise RuntimeError("EWF fragmentation has not completed successfully.")

if (
    ewf_ref is None
    or ewf_ref.run_id != RUN_ID
    or not ewf_ref.file_sha256
):
    raise RuntimeError("Verified current-run EWF artifact is missing.")

# Recover the fragment data for subsequent scientific analysis.
ewf_stage2_result = lab._build_ewf_result_from_status(ewf_status)

if ewf_stage2_result is None:
    raise RuntimeError("Unable to recover verified Stage 2 results.")

print(f"Stage 2 verified: {ewf_ref.artifact_name}")


### 9A · Verify fragment provenance

Confirm that the embedded fragments originate from the verified Stage 2 calculation associated with the current experiment.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to validate the active `RUN_ID`, Stage 2 completion status, and fragment artifact identity.

</div>

**Expected outcome:** The verified fragment artifact name and current campaign identifier.

This cell is **read-only**, performs no remote calculations, and establishes the provenance required for subsequent fragment analysis.

In [ ]:
# ── 9A · Confirm fragment provenance ────────────────────────────

if not USE_FULL_LIVE_TRIAL or run_ctx is None or run_ctx.run_id != RUN_ID:
    raise RuntimeError("Current campaign is not initialized.")

if (
    ewf_ref is None
    or ewf_ref.run_id != RUN_ID
    or not ewf_ref.file_sha256
    or not ewf_status.ok
    or ewf_stage2_result is None
):
    raise RuntimeError("A verified Stage 2 result is required.")

print(f"Fragment source: {ewf_ref.artifact_name}")
print(f"Campaign: {RUN_ID}")

### 9B · Recover the verified fragment catalog

Recover the embedded fragment catalog from the completed Stage 2 EWF calculation.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to verify the fragment identifiers, orbital dimensions, and provenance of the computed fragment Hamiltonians.

</div>

The catalog records each fragment's spatial orbital count and occupied and virtual orbital spaces.

**Expected outcome:** A validated fragment catalog displaying the number of fragments, orbital-size range, and mean orbital count.

This cell is **read-only**, performs no remote calculations, and establishes `ixa` and `_n_orb` for subsequent solver allocation.

In [ ]:

# ── 9B · Recover the verified fragment catalog ───────────────────

if (
    not USE_FULL_LIVE_TRIAL
    or run_ctx is None
    or run_ctx.run_id != RUN_ID
):
    raise RuntimeError("Current campaign is not initialized.")

if (
    ewf_ref is None
    or ewf_ref.run_id != RUN_ID
    or not ewf_ref.file_sha256
    or not ewf_status.ok
    or ewf_stage2_result is None
):
    raise RuntimeError("Verified current-run Stage 2 data is required.")

# Recover the fragment catalog from the verified EWF result.
ixa = ewf_catalog_from_result(ewf_stage2_result).copy()

required = {"fragment_id", "n_orb", "n_occ", "n_vir"}

if ixa.empty or not required.issubset(ixa.columns):
    raise RuntimeError("Stage 2 fragment catalog is incomplete.")

if "ligand" in ixa.columns:
    if not ixa["ligand"].eq(MOLECULE.name).all():
        raise RuntimeError("Fragment catalog ligand mismatch.")

# Validate orbital dimensions and fragment identifiers.
for col in required:
    ixa[col] = pd.to_numeric(ixa[col], errors="raise")

    if (
        not np.isfinite(ixa[col].to_numpy(dtype=float)).all()
        or (ixa[col] % 1 != 0).any()
    ):
        raise RuntimeError(f"Invalid catalog values: {col}")

    ixa[col] = ixa[col].astype(int)

ixa = ixa.sort_values("fragment_id").reset_index(drop=True)

if (
    ixa["fragment_id"].duplicated().any()
    or (ixa["fragment_id"] < 0).any()
    or (ixa["n_orb"] <= 0).any()
    or (ixa["n_occ"] < 0).any()
    or (ixa["n_vir"] < 0).any()
    or not (
        ixa["n_occ"] + ixa["n_vir"] == ixa["n_orb"]
    ).all()
):
    raise RuntimeError("Stage 2 fragment catalog validation failed.")

_n_orb = ixa["n_orb"]

print("Fragment catalog: VERIFIED")
print(f"Run ID:         {RUN_ID}")
print(f"Fragments:      {len(ixa)}")
print(f"Orbital range:  {_n_orb.min()}–{_n_orb.max()}")
print(f"Mean orbitals:  {_n_orb.mean():.1f}")
print(f"Source:         {ewf_ref.artifact_name}")


### 9C · Inspect fragment solver distribution

Visualize the distribution of spatial orbital counts across the verified embedded fragments and examine their corresponding solver categories.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to classify the fragments according to the orbital-size thresholds separating FCI, CCSD, and quantum-assisted TrimSQD candidates.

</div>

Inspect the resulting histogram and determine how many fragments are assigned to each solver category.

**Expected outcome:** A histogram displaying the computed fragment distribution, solver boundaries, and corresponding counts.

This cell performs **no remote calculations**. The visualization describes the planned solver allocation, not computed correlation energies.

Proceed to the subsequent exercises using the fragment distribution obtained from your calculation.

In [ ]:

# ── 9C · Fragment solver distribution ─────────────────────────────

from matplotlib.patches import Patch

# Validate the verified fragment orbital counts.
if _n_orb.empty or _n_orb.isna().any():
    raise RuntimeError("Missing or invalid fragment orbital counts.")

if len(_n_orb) != len(ixa):
    raise RuntimeError(
        "Orbital counts do not match the verified fragment catalog."
    )

if FCI_THRESHOLD >= SQD_THRESHOLD:
    raise ValueError("FCI_THRESHOLD must be below SQD_THRESHOLD.")

# Classify fragments by spatial orbital count.
_n_fci = int((_n_orb <= FCI_THRESHOLD).sum())
_n_ccsd = int(
    ((_n_orb > FCI_THRESHOLD) & (_n_orb < SQD_THRESHOLD)).sum()
)
_n_qpu = int((_n_orb >= SQD_THRESHOLD).sum())

# Verify that every fragment receives exactly one classification.
if _n_fci + _n_ccsd + _n_qpu != len(ixa):
    raise RuntimeError(
        "Solver classification does not cover the fragment catalog."
    )

# Count fragments by orbital size.
sizes = np.arange(_n_orb.min(), _n_orb.max() + 1)
counts = _n_orb.value_counts().reindex(sizes, fill_value=0).to_numpy()

colors = [
    BLUE if n <= FCI_THRESHOLD
    else GRAY if n < SQD_THRESHOLD
    else ORANGE
    for n in sizes
]

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(
    sizes, counts,
    width=0.85,
    color=colors,
    edgecolor="white",
    zorder=3,
)

# Show solver boundaries.
for boundary in (FCI_THRESHOLD + 0.5, SQD_THRESHOLD - 0.5):
    ax.axvline(boundary, ls="--", lw=1, alpha=0.5)

# Label populated bins.
for bar, count in zip(bars, counts):
    if count:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            count + 0.1,
            str(int(count)),
            ha="center",
            va="bottom",
        )

ax.legend(
    handles=[
        Patch(facecolor=BLUE, label=f"FCI · {_n_fci}"),
        Patch(facecolor=GRAY, label=f"CCSD · {_n_ccsd}"),
        Patch(facecolor=ORANGE, label=f"TrimSQD · {_n_qpu}"),
    ],
    frameon=False,
    ncol=3,
)

ax.set(
    title=f"{MOLECULE.name.capitalize()} embedded fragment solver sizing",
    xlabel="Spatial orbitals per fragment",
    ylabel="Number of fragments",
    xticks=sizes,
)
ax.set_ylim(0, counts.max() * 1.25 + 1)
ax.grid(axis="y", alpha=0.15)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()

_plot_path = (
    lab.benchmark_path.parent
    / f"{MOLECULE.name}_fragment_solver_distribution.png"
)

fig.savefig(_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved plot: {_plot_path}")
print(
    f"Total: {len(ixa)}  |  "
    f"FCI: {_n_fci}  |  "
    f"CCSD: {_n_ccsd}  |  "
    f"TrimSQD: {_n_qpu}"
)


## 3 · Assign correlated fragment solvers

The hybrid workflow combines two classical electronic-structure methods:

- **Full configuration interaction (FCI):** Exact diagonalization within a finite fragment Hamiltonian, with computational cost increasing rapidly with system size.
- **Coupled-cluster singles and doubles (CCSD):** An approximate correlated method using single and double excitations.

**Reference:** [Bartlett and Musiał (2007), §III](https://doi.org/10.1103/RevModPhys.79.291).

Solver selection follows the fragment orbital counts:

| Fragment size | Solver |
|---|---|
| At most 10 spatial orbitals | FCI |
| 11–19 spatial orbitals | CCSD |
| At least 20 spatial orbitals | Quantum-assisted TrimSQD |

<div class="alert alert-block alert-success">

✏️ **Your task:** Run **10A–10B** to verify the current-run fragment orbital counts and inspect the scientific solver configurations and their digest.

</div>

Using the orbital-size distribution obtained in Section 2, determine how many fragments fall into each solver category.

The quantum-fragment selection is examined in Section 4. Final solver assignments are constructed and frozen into the `RunManifest` in **E.8**.

**Expected outcome:** Verified fragment sizes and solver configurations, with the resulting solver allocation determined from your calculation.

This section performs **no remote calculations**. Correlation energies are computed in subsequent stages.

### 10A · Verify fragment orbital counts

Prepare the orbital-count mapping used to assign correlated fragment solvers.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to verify the current-run Stage 2 artifact and validate the fragment catalog.

</div>

The cell checks fragment identifiers, positive orbital counts, and duplicate entries, then constructs `_norb_map`.

**Expected outcome:** A validated mapping of fragment identifiers to their computed spatial orbital counts.

This cell **performs no remote calculations**. Proceed to **10B** to configure the correlated solvers.

In [ ]:
# ── 10A · Prepare fragment orbital counts ─────────────────────────

if not USE_FULL_LIVE_TRIAL or run_ctx is None:
    raise RuntimeError("Initialize the full campaign first.")

_stage2_ref = load_stage_ref(run_ctx, "embedding_ewf")

if _stage2_ref is None or _stage2_ref.run_id != RUN_ID:
    raise RuntimeError("Current-run Stage 2 artifact is required.")

if not ewf_status.ok:
    raise RuntimeError("Stage 2 verification failed.")

if ixa is None or ixa.empty:
    raise RuntimeError("Fragment catalog is unavailable.")

_required_columns = {"fragment_id", "n_orb"}

if not _required_columns.issubset(ixa.columns):
    raise RuntimeError("Fragment catalog is incomplete.")

_norb_map = dict(zip(
    ixa["fragment_id"].astype(int),
    ixa["n_orb"].astype(int),
))

if len(_norb_map) != len(ixa):
    raise RuntimeError("Duplicate fragment identifiers.")

if any(n <= 0 for n in _norb_map.values()):
    raise RuntimeError("Invalid orbital counts.")

print(f"Verified {len(_norb_map)} fragments from the current campaign.")

### 10B · Configure correlated fragment solvers

Establish the scientific solver configurations for FCI, CCSD, SQD, and TrimSQD.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to initialize `SOLVER_CONFIG`, inspect the registered solvers, and generate the configuration digest.

</div>

**Expected outcome:** A list of registered solvers and a reproducible `SOLVER_CONFIG` digest.

This cell **does not assign fragments or execute calculations**. Final solver assignments are established in **E.8**, while the quantum solver settings are refined in Section 5.

In [ ]:
# ── 10B · Scientific solver configurations ───────────────────────

SOLVER_CONFIG = Stage3Config(
    fci=FCIConfig(),
    ccsd=CCSDConfig(),
    sqd=SQDConfig(),
    trimsqd=TrimSQDConfig(),
)

print("Registered solvers:", list_registered_solvers())
print("Solver configuration digest:", SOLVER_CONFIG.digest())

## 4 · Select quantum-treated fragments

The hybrid workflow assigns quantum-assisted solvers to fragments containing **at least 20 spatial orbitals**:

$$
n_{\mathrm{orb}} \geq 20.
$$

This criterion selects fragments by orbital size, rather than by a measured ranking of correlation difficulty.

<div class="alert alert-block alert-success">

✏️ **Your task:** Complete `MIN_N_ORB` using the specified orbital threshold, then execute the selection cell.

</div>

Inspect `FINAL_FRAGMENT_IDS` and verify that every selected fragment satisfies the quantum-eligibility criterion.

Proceed to **11B** to examine the orbital properties of the selected fragments.

**Expected outcome:** A verified list of quantum-eligible fragment identifiers derived from the current-run Stage 2 catalog.

Selection is **read-only** and performs no remote calculations. The selected fragments will be used in the SQD/TrimSQD laboratory and the full hybrid campaign.

In [ ]:

# ── 11A · Select quantum fragments ───────────────────────────────

MIN_N_ORB = FILL_IN_MIN_N_ORB
_mol_name = MOLECULE.name

if ixa is None or ixa.empty or not ewf_status.ok:
    raise RuntimeError("Verified Stage 2 fragment data is required.")

FINAL_FRAGMENT_IDS = sorted(
    ixa.loc[ixa["n_orb"] >= MIN_N_ORB, "fragment_id"]
    .astype(int)
    .tolist()
)

final_selection = {
    "selected_fragment_ids": FINAL_FRAGMENT_IDS,
    "ligand": _mol_name,
    "min_n_orb": MIN_N_ORB,
}

print(
    f"Selected {len(FINAL_FRAGMENT_IDS)} quantum fragments "
    f"with at least {MIN_N_ORB} spatial orbitals."
)
print(f"Fragment IDs: {FINAL_FRAGMENT_IDS}")


In [ ]:
# ── 11B · Inspect quantum fragment selection ─────────────────────

_disp_cols = [
    col for col in (
        "fragment_id", "n_orb", "n_occ", "n_vir", "sqd_n_qubits"
    )
    if col in ixa.columns
]

_sel_df = (
    ixa.loc[ixa["fragment_id"].isin(FINAL_FRAGMENT_IDS), _disp_cols]
    .sort_values("fragment_id")
)

display(_sel_df)

### D.1 · Verify quantum-fragment selection

Validate the quantum-fragment selection derived from the current-run Stage 2 calculation.

<div class="alert alert-block alert-success">

✏️ **Your task:** Confirm that the fragments selected in **11A** satisfy your orbital-size criterion and that their orbital properties agree with the verified fragment catalog.

</div>

The cell checks the current-run Stage 2 provenance, selected fragment identifiers, and orbital data without modifying the selection.

**Expected outcome:** A verified selection and a table displaying the fragment identifiers and orbital dimensions.

This cell is **read-only** and performs no remote calculations. The verified selection is used for Exercise 2 and the subsequent single-fragment laboratory.

In [ ]:

# ── D.1 · Verify quantum-fragment selection ──────────────────────

if ixa is None or ixa.empty or not ewf_status.ok:
    raise RuntimeError("Verified Stage 2 fragment data is required.")

if (
    ewf_ref is None
    or ewf_ref.run_id != RUN_ID
    or not ewf_ref.file_sha256
):
    raise RuntimeError("Current-run Stage 2 artifact is required.")

# Independently verify the selection established in 11A.
_expected_ids = sorted(
    ixa.loc[ixa["n_orb"] >= MIN_N_ORB, "fragment_id"]
    .astype(int)
    .tolist()
)

if not FINAL_FRAGMENT_IDS or FINAL_FRAGMENT_IDS != _expected_ids:
    raise RuntimeError(
        "Quantum-fragment selection does not match the orbital threshold."
    )

if (
    final_selection["selected_fragment_ids"] != FINAL_FRAGMENT_IDS
    or final_selection["ligand"] != MOLECULE.name
    or final_selection["min_n_orb"] != MIN_N_ORB
):
    raise RuntimeError("Fragment-selection metadata is inconsistent.")

# Verify the orbital table produced in 11B.
_expected_df = (
    ixa.loc[
        ixa["fragment_id"].isin(FINAL_FRAGMENT_IDS),
        _sel_df.columns,
    ]
    .sort_values("fragment_id")
    .reset_index(drop=True)
)

if not _sel_df.reset_index(drop=True).equals(_expected_df):
    raise RuntimeError("Selected fragment orbital data is inconsistent.")

print("Quantum-fragment selection: VERIFIED")
print(f"Selected fragments: {len(FINAL_FRAGMENT_IDS)}")
print(f"Orbital threshold: {MIN_N_ORB}")
display(_sel_df)


### D.2 · Publish the verified fragment catalog

Publish the fragment catalog constructed from your completed Stage 2 calculation.

<div class="alert alert-block alert-success">

✏️ **Your task:** Verify that the fragment identifiers, orbital counts, and occupied orbital spaces are consistent with the current-run EWF results.

</div>

The cell prepares the catalog in the format required by downstream selection utilities and writes it to `challenge_data/fragment_catalog.csv`.

**Expected outcome:** A published fragment catalog containing your computed results.

This cell performs **local file operations only**. It does not submit remote calculations or QPU jobs.

In [ ]:

# ── D.2 · Publish the verified fragment catalog ──────────────────

if ixa is None or ixa.empty or not ewf_status.ok:
    raise RuntimeError("Verified Stage 2 fragment data is required.")

if (
    ewf_ref is None
    or ewf_ref.run_id != RUN_ID
    or not ewf_ref.file_sha256
):
    raise RuntimeError("Current-run Stage 2 artifact is required.")

# Construct the downstream catalog from verified Stage 2 data.
catalog = pd.DataFrame({
    "ligand": MOLECULE.name,
    "fragment_id": ixa["fragment_id"].astype(int),
    "n_orb": ixa["n_orb"].astype(int),
    "n_elec": 2 * ixa["n_occ"].astype(int),
})

catalog = (
    catalog.sort_values("fragment_id")
    .reset_index(drop=True)
)

if (
    catalog.empty
    or catalog["fragment_id"].duplicated().any()
    or (catalog["n_orb"] <= 0).any()
    or (catalog["n_elec"] < 0).any()
    or (catalog["n_elec"] > 2 * catalog["n_orb"]).any()
):
    raise RuntimeError("Invalid fragment catalog.")

# Publish for downstream fragment-selection utilities.
lab.cat_path.parent.mkdir(parents=True, exist_ok=True)
catalog.to_csv(lab.cat_path, index=False)

print(f"Published {len(catalog)} fragments → {lab.cat_path}")
print(f"Source run: {RUN_ID}")


In [ ]:
# ── D.3 · Freeze the final quantum-fragment selection ──

final_selection = lab.select_fragments_by_size(
    ligand=MOLECULE.name,
    min_n_orb=MIN_N_ORB,
)

print(f"Frozen quantum selection: {FINAL_FRAGMENT_IDS}")

### Exercise 2 · Submit the selected fragments

Submit the quantum-eligible fragments identified from your EWF calculation.

<div class="alert alert-block alert-success">

✏️ **Your task:** Inspect `_sel_df` and verify that your fragment selection satisfies the orbital-size criterion established in Section 4.

</div>

The submission bundle is constructed from your computed results and contains:

- `fragment_count` — Number of selected fragments.
- `fragments` — Fragment identifiers and their `n_orb`, `n_occ`, and `n_vir` values.

When satisfied with your selection, set `SUBMIT_EXERCISE_2 = True` to submit your results.

**Expected outcome:** Grader feedback on your selected fragments and orbital dimensions.

Submission is disabled by default. This cell performs no molecular calculations or QPU sampling.

In [ ]:

# ── Exercise 2 · EWF fragment selection ──────────────────────────

SUBMIT_EXERCISE_2 = False

if SUBMIT_EXERCISE_2:
    if "_sel_df" not in globals() or _sel_df.empty:
        raise RuntimeError(
            "Complete fragment selection before submitting Exercise 2."
        )

    bundle = {
        "fragment_count": len(_sel_df),
        "fragments": {
            str(int(row.fragment_id)): {
                "n_orb": int(row.n_orb),
                "n_occ": int(row.n_occ),
                "n_vir": int(row.n_vir),
            }
            for row in _sel_df.itertuples(index=False)
        },
    }

    grade_lab_proteasome_ex2(bundle)

else:
    print("Exercise 2 submission disabled.")


## Exercise 3

## 5 · Single-fragment SQD and TrimSQD laboratory

**Sample-based quantum diagonalization (SQD)** constructs electronic configuration spaces from sampled occupation-number bitstrings, then classically diagonalizes the molecular Hamiltonian within those spaces. The bitstrings are not direct energy measurements.

**TrimSQD** extends SQD using a two-level configuration-selection procedure.

**References:** [Robledo-Moreno et al. (2025), Fig. 1 and Eq. (3)](https://doi.org/10.1126/sciadv.adu9991) · [Merz et al. (2026), Fig. 2](https://arxiv.org/abs/2605.01138)

### Exercise 3 · Compare CCSD, SQD, and TrimSQD

**Objective:** Compute and compare the energies of the same embedded fragment using classical CCSD and quantum-sampled SQD/TrimSQD.

**01 · Select and configure the fragment**

In **D.2**, establish `TARGET_FRAGMENT_ID` for the experimental fragment.

In **E.2**, configure the experimental settings:

- `N_SHOTS`, `N_REPS` — Quantum measurement acquisition.
- `SQD_CONFIG` — Configuration sampling, diagonalization iterations, and subspace dimensions.
- `TRIM_CONFIG` — Configuration trimming and carryover.

All three solvers must use the **same verified Stage 2 fragment Hamiltonian**.

**02 · Execute the single-fragment laboratory**

Run the following cells sequentially, enabling execution deliberately when ready:

| Cell | Operation |
|---|---|
| E.3a.0 | Initialize the fragment laboratory |
| E.3a.1 | Submit the CCSD reference |
| E.3a.2 | Retrieve CCSD energy and Hamiltonian fingerprint |
| E.3b.1 | Submit the SQD/TrimSQD experiment |
| E.3b.2 | Retrieve energies and verify Hamiltonian identity |

Execution flags are initially disabled.

Set `LIVE_QPU_SAMPLING = True` only when deliberately authorizing live QPU acquisition. Otherwise, the laboratory uses the configured measurement-replay pathway.

<div class="alert alert-block alert-success">

✏️ **Your task:** Examine `e_ccsd`, `e_sqd`, and `e_trim`. Compare the quantum-assisted energies against CCSD and investigate the differences.

</div>

The Hamiltonian fingerprint check ensures that the comparison concerns the same electronic problem.

**03 · Submit the three energies**

Inspect the computed energies and complete the Exercise 3 submission cell.

When ready, enable submission and call:

`grade_lab_proteasome_ex3(bundle)`

**Grading · Exercise 3**

| Solver | Energy tolerance |
|---|---:|
| CCSD | ±0.001 Ha |
| SQD | ±0.020 Ha |
| TrimSQD | ±0.020 Ha |

**If grading fails:** Verify fragment identity, Hamiltonian consistency, measurement provenance, and solver configurations before resubmitting.

**Outcome:** A validated comparison of classical and quantum-sampled fragment solvers, establishing the foundation for the tuning experiment in Exercise 4.

### D.2 · Select the experimental fragment

Select the fragment used in the single-fragment SQD/TrimSQD laboratory.

<div class="alert alert-block alert-success">

✏️ **Your task:** Use **fragment 1** for the graded experiment.

</div>

The Exercise 3 and Exercise 4 grading references are defined for this designated fragment. Other quantum-eligible fragments may be explored independently, but they are not used for the graded submission pathway.

The cell verifies that fragment 1 is quantum-eligible under your current fragment selection.

**Expected outcome:** The selected experimental fragment identifier, validated against `FINAL_FRAGMENT_IDS`.

This cell **defines the experimental target only** and performs no remote calculations. Proceed to **E.2** to configure SQD and TrimSQD.

In [ ]:
# ── D.2 · Select the experimental fragment ───────────────────────

TARGET_FRAGMENT_ID = 1
ACTIVE_FRAGMENT_ID = int(TARGET_FRAGMENT_ID)

if ACTIVE_FRAGMENT_ID not in FINAL_FRAGMENT_IDS:
    raise ValueError(
        f"Fragment {ACTIVE_FRAGMENT_ID} is not quantum-eligible."
    )

print(f"Experimental fragment: {MOLECULE.name} / {ACTIVE_FRAGMENT_ID}")

### E.2 · Configure SQD and TrimSQD

Specify the quantum acquisition and classical post-processing parameters for the single-fragment laboratory.

<div class="alert alert-block alert-success">

✏️ **Your task:** Complete the `FILL_IN_*` placeholders for the SQD and TrimSQD solvers.

</div>

**SQD configuration**

- `n_batches` — Number of independently constructed configuration batches.
- `iterations` — Number of self-consistent diagonalization iterations.
- `samples_per_batch` — Number of sampled configurations per batch.
- `max_dim` — Maximum configuration subspace dimension.

**TrimSQD configuration**

- `trim_ratio` — Fraction of sampled configurations retained during trimming.

The remaining parameters establish numerical tolerances, reproducibility, and configuration-management limits.

`N_SHOTS` inherits the measurement budget you selected in **0B**. `N_REPS` specifies the prescribed circuit repetition count.

Choose parameters consistent with the computational budget and the challenge's scientific protocol. The resulting energies depend on these settings, so use the same configuration when comparing and validating your results.

**Expected outcome:** Printed acquisition settings and complete SQD/TrimSQD configuration dictionaries.

This cell **defines settings only** and performs no calculations or QPU sampling. Proceed to **E.3a.0** to initialize the fragment laboratory.

In [ ]:

# ── E.2 · SQD and TrimSQD configurations ─────────────────────────

# Inherit the participant-selected QPU shot budget from 0B.
N_SHOTS = FULL_TRIAL_SHOTS

# Fixed circuit repetition count for this challenge.
N_REPS = 1

SQD_CONFIG = {
    "n_batches": FILL_IN_N_BATCHES,
    "iterations": FILL_IN_SQD_ITERATIONS,
    "samples_per_batch": FILL_IN_SAMPLES_PER_BATCH,
    "max_dim": FILL_IN_MAX_DIM,
    "energy_tol": 1e-8,
    "occupancies_tol": 1e-5,
    "symmetrize_spin": True,
    "seed": 20260626,
    "carryover_threshold": 1e-4,
}

TRIM_CONFIG = {
    "trim_ratio": FILL_IN_TRIM_RATIO,
    "max_strings_per_trim": 300,
    "max_carryover": 400,
    "sector_trim": False,
}

print(f"QPU acquisition: {N_SHOTS:,} shots × {N_REPS} repetition(s)")
print("SQD:", SQD_CONFIG)
print("TrimSQD:", TRIM_CONFIG)


### E.3a.0 · Initialize fragment laboratory

Prepare the single-fragment laboratory using the verified Hartree–Fock reference, Stage 2 embedded fragments, and current-run artifacts.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to initialize `fragment_lab` and establish access to the selected fragment's Hamiltonian.

</div>

**Expected outcome:** `Fragment laboratory ready`, followed by the current `RUN_ID`.

This cell prepares the laboratory without submitting remote calculations. Proceed to **E.3a.1** to establish the independent CCSD reference.

In [ ]:
# ── E.3a.0 · Initialize fragment laboratory ──────────────────────

from proteasome_challenge.fragment_lab import prepare_fragment_lab

fragment_lab = prepare_fragment_lab(
    lab, run_ctx, mf_ref, ewf_ref,
    FINAL_FRAGMENT_IDS, ixa, MOLECULE.name,
)

print(f"Fragment laboratory ready · {RUN_ID}")

### E.3a.1 · Submit CCSD reference

Establish an independent classical CCSD benchmark for the designated embedded fragment.

<div class="alert alert-block alert-success">

✏️ **Your task:** Verify that the fragment laboratory is initialized and `ACTIVE_FRAGMENT_ID` identifies fragment 1. When ready, set `RUN_CCSD_REFERENCE = True` to submit the CCSD calculation.

</div>

The calculation uses the verified Stage 2 fragment Hamiltonian and records its Serverless job identifier.

**Expected outcome:** `CCSD job`, followed by the submitted job ID.

**Execution is disabled by default.** Enabling this cell may submit a remote classical calculation, but performs no QPU sampling.

Proceed to **E.3a.2** to retrieve the CCSD energy and verify its Hamiltonian fingerprint.

In [ ]:

# ── E.3a.1 · Submit CCSD reference ───────────────────────────────

RUN_CCSD_REFERENCE = False

if RUN_CCSD_REFERENCE:
    ccsd_submission = fragment_lab.submit_fragment(
        MOLECULE.name,
        ACTIVE_FRAGMENT_ID,
        method="ccsd",
        execution="replay",
    )

    print("CCSD job:", ccsd_submission["job_id"])


### E.3a.2 · Retrieve CCSD reference

Retrieve the completed CCSD calculation for fragment 1.

<div class="alert alert-block alert-success">

✏️ **Your task:** Once the calculation submitted in E.3a.1 has completed, set `RETRIEVE_CCSD_REFERENCE = True` and execute this cell.

</div>

The cell retrieves the CCSD energy and its Hamiltonian SHA-256 fingerprint. These establish the classical reference and verify that subsequent SQD and TrimSQD calculations use the same fragment Hamiltonian.

**Expected outcome:** `Checkpoint 07`, displaying `e_ccsd` and `ccsd_hash`.

Retrieval is **disabled by default**. Enabling it may poll an existing remote job but does not submit new calculations or perform QPU sampling.

Proceed to **E.3b.1** to execute SQD and TrimSQD.

In [ ]:

# ── E.3a.2 · Retrieve CCSD reference → Checkpoint 07 ────────────

RETRIEVE_CCSD_REFERENCE = False

if RETRIEVE_CCSD_REFERENCE:
    ccsd_result = fragment_lab.retrieve_fragment(
        MOLECULE.name,
        ACTIVE_FRAGMENT_ID,
        method="ccsd",
        poll_seconds=POLL_SECONDS,
    )

    e_ccsd = fragment_lab.energy_of(ccsd_result, "ccsd")
    ccsd_hash = ccsd_result.get("hamiltonian_sha256")

    if e_ccsd is None or not ccsd_hash:
        raise RuntimeError("Incomplete CCSD reference result.")

    print(f"Checkpoint 07 · CCSD: {e_ccsd:.10f} Ha")
    print(f"Hamiltonian SHA-256: {ccsd_hash}")


### E.3b.1 · Submit SQD and TrimSQD experiment

Execute SQD and TrimSQD on the designated embedded fragment using your configured solver parameters and quantum measurements.

<div class="alert alert-block alert-success">

✏️ **Your task:** After retrieving the CCSD reference in E.3a.2, review your experimental settings and deliberately enable execution.

</div>

- `RUN_FRAGMENT_LAB = True` — Authorizes submission of the fragment experiment.
- `LIVE_QPU_SAMPLING = True` — Selects live quantum measurement acquisition; configure this in **0A**.
- `LIVE_QPU_SAMPLING = False` — Selects replay using previously acquired measurements.
- `N_SHOTS`, `N_REPS`, and `QPU_BACKEND` — Define the quantum acquisition settings.

**Expected outcome:** A submitted SQD/TrimSQD experiment with a recorded job ID.

**Execution is disabled by default.** Enabling this stage may consume QPU and Serverless resources. Replay requires suitable existing measurement data.

Proceed to **E.3b.2** to retrieve the energies and verify Hamiltonian identity.

In [ ]:

# ── E.3b.1 · Submit SQD / TrimSQD experiment ────────────────────

RUN_FRAGMENT_LAB = False

if RUN_FRAGMENT_LAB:
    if not globals().get("ccsd_hash"):
        raise RuntimeError("Retrieve the CCSD reference first.")

    fragment_submission = fragment_lab.submit_fragment(
        MOLECULE.name,
        ACTIVE_FRAGMENT_ID,
        method="both",
        execution="live_qpu" if LIVE_QPU_SAMPLING else "replay",
        sqd_config=SQD_CONFIG,
        trim_config=TRIM_CONFIG,
        n_shots=N_SHOTS,
        n_reps=N_REPS,
        qpu_backend=QPU_BACKEND,
        allow_second_qpu=ALLOW_SECOND_QPU_SAMPLING,
    )

    print("SQD/TrimSQD job:", fragment_submission["job_id"])


### E.3b.2 · Retrieve SQD and TrimSQD results

Retrieve the quantum-assisted fragment calculations and compare their energies against the independent CCSD reference.

<div class="alert alert-block alert-success">

✏️ **Your task:** Once the experiment submitted in **E.3b.1** has completed, set `RETRIEVE_FRAGMENT_LAB = True` and execute this cell.

</div>

The cell retrieves the SQD and TrimSQD energies, verifies that all three solvers used the same fragment Hamiltonian, and reports their energy differences relative to CCSD.

**Expected outcome:** `Checkpoint 08`, displaying `e_sqd`, `e_trim`, their deviations from `e_ccsd`, and `Hamiltonian identity: VERIFIED`.

Retrieval is **disabled by default**. Enabling it may poll an existing remote job but does not submit new calculations or perform QPU sampling.

Proceed to the Exercise 3 submission after verifying the results.

In [ ]:

# ── E.3b.2 · Retrieve SQD / TrimSQD → Checkpoint 08 ─────────────

RETRIEVE_FRAGMENT_LAB = False

if RETRIEVE_FRAGMENT_LAB:
    if not globals().get("ccsd_hash"):
        raise RuntimeError("Retrieve the CCSD reference first.")

    frag_job_result = fragment_lab.retrieve_fragment(
        MOLECULE.name,
        ACTIVE_FRAGMENT_ID,
        method="both",
        poll_seconds=POLL_SECONDS,
    )

    if frag_job_result.get("hamiltonian_sha256") != ccsd_hash:
        raise RuntimeError("CCSD and SQD/TrimSQD Hamiltonians differ.")

    e_sqd = fragment_lab.energy_of(frag_job_result, "sqd")
    e_trim = fragment_lab.energy_of(frag_job_result, "trimsqd")

    if e_sqd is None or e_trim is None:
        raise RuntimeError("Incomplete SQD/TrimSQD result.")

    print(f"Checkpoint 08 · SQD:  {e_sqd:.10f} Ha")
    print(f"TrimSQD:               {e_trim:.10f} Ha")
    print(f"CCSD reference:        {e_ccsd:.10f} Ha")
    print(f"SQD − CCSD:            {e_sqd - e_ccsd:+.8f} Ha")
    print(f"TrimSQD − CCSD:        {e_trim - e_ccsd:+.8f} Ha")
    print("Hamiltonian identity: VERIFIED")


### Exercise 3 · Submit single-fragment solver energies

Compare the energies obtained for the designated fragment using CCSD, SQD, and TrimSQD.

<div class="alert alert-block alert-success">

✏️ **Your task:** Inspect the results from E.3a.2 and E.3b.2. Confirm that all three calculations use the same verified fragment Hamiltonian.

</div>

The submission bundle contains:

- `single_fragment_ccsd_energy` — Classical CCSD reference.
- `single_fragment_sqd_energy` — SQD energy.
- `single_fragment_trimsqd_energy` — TrimSQD energy.

Review the computed energies and their differences. When satisfied, set `SUBMIT_EXERCISE_3 = True`.

**Expected outcome:** Grader feedback on the three fragment energies.

Grading submission is **disabled by default**. This cell performs no scientific calculations or QPU sampling.

In [ ]:

# ── Exercise 3 · Single-fragment solver energies ────────────────

SUBMIT_EXERCISE_3 = False

if SUBMIT_EXERCISE_3:
    if not all(
        name in globals()
        for name in ("e_sqd", "e_trim", "e_ccsd")
    ):
        raise RuntimeError(
            "Complete the CCSD, SQD, and TrimSQD experiments "
            "before submitting Exercise 3."
        )

    bundle = {
        "single_fragment_sqd_energy": float(e_sqd),
        "single_fragment_trimsqd_energy": float(e_trim),
        "single_fragment_ccsd_energy": float(e_ccsd),
    }

    grade_lab_proteasome_ex3(bundle)
else:
    print("Exercise 3 submission disabled.")


## Exercise 4

### E.5 · SQD and TrimSQD tuning

The accuracy and computational cost of SQD and TrimSQD depend on the configuration subspace, diagonalization settings, and trimming strategy.

Using the **same fragment Hamiltonian and QPU measurement data**, a tuning grid compares eight solver configurations to investigate these tradeoffs.

**Reference:** [Robledo-Moreno et al. (2025)](https://doi.org/10.1126/sciadv.adu9991)

### Exercise 4 · Select the optimal solver configuration

**Objective:** Evaluate the tuning-grid results and identify the lowest-energy configuration for promotion to the full molecular calculation.

**01 · Execute the tuning grid**

Run **E.3d** with `RUN_TUNING_GRID = True` to execute or resume the eight configurations.

The workflow reuses existing measurements and recorded results, resumes incomplete work, and verifies Hamiltonian consistency with the CCSD reference.

Use the read-only monitoring cell **E.3c** to inspect remote execution progress when necessary. No additional QPU sampling is required.

**02 · Analyze the results**

Run **E.3e** to inspect `grid_df` and compare fragment energies against computational cost.

Examine:

- `energy_ha` — computed fragment energy
- `diag_calls` — classical diagonalization count
- `max_dim` — maximum configuration subspace dimension
- `trim_ratio` — TrimSQD trimming parameter
- `solver_method` — SQD or TrimSQD

**Questions to investigate:**

- Which configuration produces the lowest energy?
- How do subspace dimension and trimming affect accuracy?
- What computational cost accompanies the improvement?
- How do the results compare with CCSD?

**03 · Submit the optimal configuration**

Set `best_config` to the label of the lowest-energy configuration.

The submission includes all eight computed energies and the selected configuration:

`grade_lab_proteasome_ex4(bundle)`

The selected configuration is subsequently transferred into `SELECTED_GRID_LABEL` during **E.6 · Protocol promotion**.

**Grading · Exercise 4**

| Criterion | Requirement |
|---|---|
| Grid completeness | All 8 configurations |
| Energy agreement | ±0.1 Ha per configuration |
| Configuration selection | Correct lowest-energy label |

**If grading fails:** Inspect the identified configurations, verify their solver settings and energies, and resubmit.

**Outcome:** A validated solver configuration ready for promotion into the 11-fragment quantum campaign.

In [ ]:
# ── E.3c · SQD / TrimSQD tuning grid → Checkpoint 09 ────────────

from typing import Any


RUN_TUNING_GRID = False

grid_results = None
grid_df = None
answer_09 = None

if RUN_TUNING_GRID:
    grid_results: list[dict[Any, Any]] = fragment_lab.run_tuning_grid(
        execution="replay",
        ligand=MOLECULE.name,
        fragment_id=ACTIVE_FRAGMENT_ID,
        n_shots=N_SHOTS,
        n_reps=N_REPS,
        qpu_backend=QPU_BACKEND,
        poll_seconds=POLL_SECONDS,
        
    )

    if any(r.get("hamiltonian_sha256") != ccsd_hash for r in grid_results):
        raise RuntimeError("Tuning grid and CCSD Hamiltonians differ.")

    print(f"Completed {len(grid_results)} tuning configurations.")

### E.3c · Monitor tuning-grid execution

Inspect the remote Serverless jobs, completed checkpoints, and solver iteration logs for the eight tuning configurations.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to monitor execution progress or diagnose worker failures.

</div>

Set `INSPECT_LABEL = None` to inspect all configurations, or specify a label to inspect one.

The cell is **read-only**, submits no calculations, and can be rerun safely while remote jobs execute.

Return to **E.3d** to resume the tuning grid and collect completed results.

In [ ]:
# ── E.3c · Monitor tuning-grid jobs (read-only) ───────────────────

import json

INSPECT_LABEL = None  # None = all eight configurations
LOG_LINES = 15

ligand, fid = MOLECULE.name, ACTIVE_FRAGMENT_ID

def _load(path):
    return json.loads(path.read_text()) if path.is_file() else {}

jobs = _load(fragment_lab._grid_jobs_path(ligand, fid))
partial = _load(fragment_lab._grid_partial_path(ligand, fid))
final = _load(fragment_lab._grid_results_path(ligand, fid))

completed = set(partial) | {
    r["label"] for r in final.get("tuning_results", [])
}
labels = [p["label"] for p in fragment_lab.load_tuning_grid()]

if INSPECT_LABEL is not None and INSPECT_LABEL not in labels:
    raise ValueError(f"Unknown configuration: {INSPECT_LABEL}")

markers = ("[step3]", "[SQD iter", "[TrimSQD iter",
           "DONE", "Traceback", "ERROR", "Exception")

print(f"Tuning grid · {RUN_ID} · fragment {fid}")
print(f"Completed: {len(completed)}/{len(labels)}")

client = None

for label in labels:
    if INSPECT_LABEL not in (None, label):
        continue

    job_id = jobs.get(label)
    print(f"\n── {label} · {'CHECKPOINTED' if label in completed else 'PENDING'}")

    if not job_id:
        print("No active job record.")
        continue

    try:
        client = client or fragment_lab._connect()
        job = client.job(job_id)
        print(f"Job: {job_id} · Status: {job.status()}")

        logs = str(job.logs() or "")
        progress = [line for line in logs.splitlines()
                    if any(m in line for m in markers)]
        print("\n".join(progress[-LOG_LINES:]) or "No progress logs.")

    except Exception as exc:
        print(f"Inspection unavailable: {exc}")

### E.3d · Resume and collect tuning-grid results

Recover the existing SQD/TrimSQD tuning experiment from recorded jobs, partial checkpoints, or completed results.

<div class="alert alert-block alert-success">

✏️ **Your task:** After initiating the tuning grid in **E.3c**, set `RUN_RESUME_TUNING_GRID = True` to resume unfinished calculations or retrieve completed results.

</div>

The cell verifies that all eight configurations are present and that their Hamiltonian fingerprints match the independent CCSD reference.

**Expected outcome:** A complete `grid_df` containing the tuning configurations and their computed energies, with the minimum energy recorded as Checkpoint 09.

**Execution is disabled by default.** Enabling this cell may resume outstanding Serverless calculations but does not initiate a new tuning experiment or acquire additional QPU measurements.

Use the monitoring cell following **E.3c** to inspect progress. Once the results are complete, proceed to the tuning-grid analysis.

In [ ]:

# ── E.3d · Resume tuning grid → Checkpoint 09 ────────────────────

import json

RUN_RESUME_TUNING_GRID = False

if RUN_RESUME_TUNING_GRID:
    if not globals().get("ccsd_hash"):
        raise RuntimeError("Retrieve the CCSD reference first.")

    ligand, fid = MOLECULE.name, ACTIVE_FRAGMENT_ID

    jobs_path = fragment_lab._grid_jobs_path(ligand, fid)
    partial_path = fragment_lab._grid_partial_path(ligand, fid)
    results_path = fragment_lab._grid_results_path(ligand, fid)

    if results_path.is_file():
        execution = "prepared"
        print("Loading completed tuning grid.")

    elif jobs_path.is_file() or partial_path.is_file():
        execution = "replay"
        partial = (
            json.loads(partial_path.read_text())
            if partial_path.is_file() else {}
        )
        print(f"Resuming tuning grid: {len(partial)} checkpointed.")

    else:
        raise RuntimeError(
            "No existing tuning-grid experiment. Execute E.3c first."
        )

    grid_results = fragment_lab.run_tuning_grid(
        execution=execution,
        ligand=ligand,
        fragment_id=fid,
        n_shots=N_SHOTS,
        n_reps=N_REPS,
        qpu_backend=QPU_BACKEND,
        poll_seconds=POLL_SECONDS,
    )

    expected_labels = {
        config["label"]
        for config in fragment_lab.load_tuning_grid()
    }
    actual_labels = [result["label"] for result in grid_results]

    if (
        len(actual_labels) != len(expected_labels)
        or set(actual_labels) != expected_labels
        or any(
            result.get("hamiltonian_sha256") != ccsd_hash
            for result in grid_results
        )
    ):
        raise RuntimeError(
            "Incomplete tuning grid or Hamiltonian mismatch."
        )

    grid_df = fragment_lab.grid_table(grid_results)
    answer_09 = float(grid_df["energy_ha"].min())

    display(grid_df)
    print(f"Checkpoint 09: {answer_09:.8f} Ha")


### E.3e · Analyze SQD / TrimSQD tuning results

Compare the eight tuning configurations against the independent CCSD reference.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to visualize fragment energy versus classical computational cost, measured by the number of diagonalization calls.

</div>

Examine how SQD and TrimSQD settings affect energy and computational effort. Identify the configuration producing the lowest energy and consider its cost relative to the alternatives.

**Expected outcome:** A tuning-grid table and comparison plot showing all eight configurations and the CCSD reference.

Use the results to select `best_config` for **Exercise 4**. The plot is **read-only** and performs no additional calculations.

In [ ]:
# ── E.3e · Analyze tuning-grid results → Checkpoint 09 ───────────

import matplotlib.pyplot as plt

if grid_results is None:
    print("Tuning grid: PENDING")
else:
    grid_df = fragment_lab.grid_table(grid_results)

    if len(grid_df) != len(fragment_lab.load_tuning_grid()):
        raise RuntimeError("Tuning grid is incomplete.")
    if grid_df["energy_ha"].isna().any() or e_ccsd is None:
        raise RuntimeError("Complete grid and CCSD reference required.")

    display(
        grid_df.sort_values("label")
        .drop(columns="delta_vs_best_mHa")
    )

    fig, ax = plt.subplots(figsize=(8, 4.5))

    for method, group in grid_df.groupby("solver_method"):
        ax.scatter(
            group["diag_calls"], group["energy_ha"],
            s=60, label=method,
        )

        for row in group.itertuples():
            ax.annotate(
                row.label, (row.diag_calls, row.energy_ha),
                xytext=(6, 5), textcoords="offset points",
                fontsize=8,
            )

    ax.axhline(e_ccsd, color="gray", ls="--", label="CCSD reference")
    ax.set(
        xlabel="Classical cost proxy (diagonalization calls)",
        ylabel="Fragment energy (Ha)",
        title=f"SQD / TrimSQD tuning · Fragment {ACTIVE_FRAGMENT_ID}",
    )
    ax.grid(alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()

### E.3f · Interpreting the tuning results

The tuning plot illustrates the accuracy–cost tradeoff of SQD and TrimSQD on the selected fragment. The computed energies may remain substantially different from the CCSD reference.

**Important distinction:** This challenge uses a simplified TrimSQD implementation without the final configuration-space extension employed in the published large-scale workflow. Its results should not be interpreted as the basic accuracy limit of TrimSQD.

For comparison, [Merz et al. (2026), Figure 7](https://arxiv.org/html/2605.01138v2), examines accuracy–time tradeoffs between TrimSQD, ExtSQD, and classical solvers on fragments of the 12,635-atom trypsin complex.

**Further exploration:** Investigate configuration-space extension using the [Qiskit SQD tutorial](https://qiskit.github.io/qiskit-addon-sqd/guides/add_fermionic_excitations_to_configuration_pool.html) or the [Merz Lab implementations](https://github.com/MIQuLab/ewf-geom-opt/tree/fcidump-start/Examples/Unfragmented_SQD_from_FCIDUMP).

In [ ]:

# Exercise 4 — Tuning Grid

# Inspect the E.3e results and select the lowest-energy configuration.
# Replace None with your selected configuration label (a string).
best_config = None

SUBMIT_EXERCISE_4 = False

if SUBMIT_EXERCISE_4:
    if "grid_df" not in globals() or grid_df is None or grid_df.empty:
        raise RuntimeError(
            "Complete and analyze the tuning grid before submitting."
        )

    if best_config not in set(grid_df["label"]):
        raise ValueError(
            "Set best_config to a valid configuration label from grid_df."
        )

    bundle = {
        row["label"]: float(row["energy_ha"])
        for _, row in grid_df.iterrows()
    }

    if len(bundle) != 8:
        raise RuntimeError("Exercise 4 requires all eight configurations.")

    bundle["best_config"] = best_config

    grade_lab_proteasome_ex4(bundle)
else:
    print("Exercise 4 submission disabled.")


## Exercise 5

## 6 · Full EWF-SQD fragment campaign

The single-fragment laboratory demonstrated how SQD and TrimSQD recover correlated energies from quantum measurement data. We now extend that approach to the complete molecular calculation.

The full workflow combines **classical FCI/CCSD solvers with quantum-assisted TrimSQD**, assigning each embedded fragment to its designated solver. The resulting fragment energies and reduced density matrices provide the inputs for molecular energy reconstruction.

[Robledo-Moreno et al. (2025)](https://doi.org/10.1126/sciadv.adu9991) · [Shajan et al. (2026)](https://doi.org/10.1021/acs.jctc.6c00364)

### Exercise 5 · Scale the tuned solver to the full molecule

**Objective:** Transfer the selected TrimSQD configuration into the full EWF calculation, execute the distributed fragment campaign, and validate the resulting quantum-treated fragment energies.

**01 · Transfer the selected configuration**

**Sections E.6–E.7 · Protocol promotion**

Return to the tuning results from Exercise 4.

1. Set `SELECTED_GRID_LABEL` to the configuration you identified as optimal.
2. Use `FREEZE_PROTOCOL` to establish the shared solver protocol.
3. Inspect the resulting protocol report, confirming the selected configuration, solver parameters, and target fragments.

**Key concept:** A shared protocol allows a solver configuration established on one representative fragment to be applied consistently across multiple embedded Hamiltonians.

In full-live-trial mode, the notebook freezes its declared reference protocol. Verify that this protocol agrees with your tuning selection; changing the label alone does not redefine the reference preset.

**02 · Establish the full fragment workload**

**Section E.8 · RunManifest and fragment tasks**

The `RunManifest` records the frozen scientific inputs, solver assignments, and fragment tasks.

Use `RUN_FREEZE_MANIFEST` and `RUN_CREATE_FRAGMENT_TASKS` as directed to prepare the complete campaign.

Inspect the resulting solver allocation:

| Solver | Fragments |
|---|---:|
| FCI | 19 |
| CCSD | 12 |
| TrimSQD | 11 |
| **Total** | **42** |

Confirm that the quantum-treated fragments are assigned the shared TrimSQD protocol and that the manifest and protocol digests agree.

**03 · Execute the distributed campaign**

**Sections 6.1–6.4 · Authorization, launch, and monitoring**

The campaign consists of three principal operations:

| Phase | Participant responsibility |
|---|---|
| Prepare | Verify the manifest, acquisition settings, authorization, and launch readiness |
| Launch | Authorize and dispatch the classical and quantum fragment workloads |
| Advance | Monitor remote execution, retrieve measurements, and complete TrimSQD post-processing |

Use `LAUNCH_FULL_TRIAL` for the initial authorized submission and `ADVANCE_FULL_TRIAL` for subsequent reconciliation passes. Follow the detailed operational instructions provided in Section 6.

The classical and quantum calculations can execute concurrently. A completed QPU measurement is **not** a completed fragment calculation: the corresponding TrimSQD post-processing must also finish.

**04 · Inspect and submit the fragment energies**

When the campaign completes, inspect the recorded fragment results.

For each of the 11 quantum-treated fragments, verify that:

- The calculation completed successfully.
- The result corresponds to the intended fragment and frozen solver protocol.
- A valid final `energy_ha` is available.

Run the Exercise 5 submission cell to collect the eleven energies into a dictionary and submit them through `grade_exercise_5(bundle)`.

**Grading · Exercise 5**

| Criterion | Requirement |
|---|---|
| Fragment coverage | All 11 quantum-treated fragments |
| Fragment identity | Correct fragment IDs |
| Energy agreement | **±0.1 Ha** per fragment |
| Overall result | All 11 energies must pass |

**If grading fails:** The grader identifies the fragments with incorrect energies. Verify that the selected tuning-grid configuration was correctly transferred to the production solver settings. Inspect the affected fragment calculations, rerun them if necessary, and resubmit.

**Outcome:** You will have completed the transition from a single-fragment SQD experiment to a distributed quantum-centric molecular calculation, producing the correlated fragment results required for Exercise 6.

### Stage E.6 · Promote the tuned solver configuration

Transfer the selected TrimSQD configuration from the single-fragment tuning experiment to the full quantum-fragment campaign.

<div class="alert alert-block alert-success">

✏️ **Your task:** Review the configuration selected in Exercise 4. When satisfied, set `FREEZE_PROTOCOL = True` to promote `best_config` into the shared solver protocol.

</div>

The promotion records:

- The selected tuning-grid configuration and solver settings.
- The source fragment's verified Hamiltonian fingerprint.
- The quantum measurement acquisition provenance.
- The target fragments identified from the verified Stage 2 orbital counts.

**Expected outcome:** `Promoted configuration`, followed by the QPU backend, quantum-fragment count, and protocol provenance.

**Execution is disabled by default.** Protocol freezing records scientific configuration and provenance but does not submit remote calculations or acquire QPU measurements.

Proceed to **E.7** to verify the promoted protocol before preparing the full campaign.

In [ ]:

# ── E.6 · Promote the selected tuning configuration ────────────────

from proteasome_challenge.tuned_promotion import freeze_tuned_protocol

FREEZE_PROTOCOL = False
SELECTED_GRID_LABEL: str | None = best_config

if "manifest" not in globals():
    manifest = None
if "shared_protocol" not in globals():
    shared_protocol = None

if FREEZE_PROTOCOL:
    if not SELECTED_GRID_LABEL or not grid_results:
        raise RuntimeError(
            "Complete Exercise 4 and select a tuning configuration "
            "before freezing the protocol."
        )

    if SELECTED_GRID_LABEL not in {
        result["label"] for result in grid_results
    }:
        raise ValueError(
            "Selected configuration is absent from the tuning results."
        )

    ft.authorize_stage_execution(
        "protocol", run_ctx, allow_data_refresh=False
    )

    manifest, shared_protocol = freeze_tuned_protocol(
        lab=lab,
        fragment_lab=fragment_lab,
        run_ctx=run_ctx,
        ligand=MOLECULE.name,
        fragment_id=ACTIVE_FRAGMENT_ID,
        fragment_ids=FINAL_FRAGMENT_IDS,
        grid_results=grid_results,
        selected_label=SELECTED_GRID_LABEL,
        ccsd_hash=ccsd_hash,
        qpu_backend=QPU_BACKEND,
        n_shots=N_SHOTS,
        n_reps=N_REPS,
    )

    print(f"Promoted configuration: {SELECTED_GRID_LABEL}")
    print(f"QPU backend: {QPU_BACKEND}")
    print(f"Quantum fragments: {len(shared_protocol['targets'])}")
    print(ft.protocol_provenance(lab.shared_protocol_report()))

else:
    print("Protocol promotion: PENDING")


### Stage E.7 · Verify the shared solver protocol

Verify that the configuration selected in the SQD/TrimSQD tuning experiment was correctly promoted to the full quantum-fragment campaign.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to confirm:

- **Configuration:** SQD and TrimSQD settings match the selected tuning-grid result.
- **Provenance:** The source fragment's Hamiltonian fingerprint matches the CCSD reference and tuning experiment.
- **Fragment selection:** The protocol targets exactly the quantum-treated fragments identified from the verified Stage 2 orbital counts.
- **Integrity:** The protocol and document digests are available for subsequent manifest verification.

</div>

**Expected outcome:** `Shared protocol: VERIFIED`, followed by the promoted configuration, provenance, digests, and target-fragment table.

This cell is **read-only** and performs no remote calculations. Each target retains its own fragment Hamiltonian while using the promoted solver settings.

The protocol must already exist before verification can succeed.

Proceed to **E.8** to freeze the `RunManifest`.

In [ ]:
# ── E.7 · Verify the promoted shared protocol ──────────────────────

shared_protocol = lab.load_shared_protocol(require_complete=True)
source = shared_protocol["source"]
settings = shared_protocol["transferable"]

selected = next(
    r for r in grid_results
    if r["label"] == SELECTED_GRID_LABEL
)

# Verify selection, provenance, and scientific consistency.
if not (
    source["fragment_id"] == ACTIVE_FRAGMENT_ID
    and source["grid_label"] == SELECTED_GRID_LABEL
    and source["validated_result"]["source"] == "tuning_results"
    and source["grid_energy_ha"] == selected["energy_ha"]
    and source["hamiltonian_sha256"]
        == selected["hamiltonian_sha256"] == ccsd_hash
    and sorted(shared_protocol["target_fragment_ids"])
        == sorted(FINAL_FRAGMENT_IDS)
    and settings["sqd_config"] == selected["sqd_config"]
    and settings["trim_config"] == selected["trim_config"]
):
    raise RuntimeError("Promoted protocol validation failed.")

print("Shared protocol: VERIFIED")
print(f"Configuration: {source['grid_label']}")
print(f"Quantum fragments: {len(shared_protocol['targets'])}")
print(f"Protocol digest: {shared_protocol['protocol_digest']}")
print(f"Document digest: {shared_protocol['document_digest']}")
print(ft.protocol_provenance(lab.shared_protocol_report()))

display(
    pd.DataFrame(shared_protocol["targets"])
    .sort_values("fragment_id")
    .reset_index(drop=True)
)

### Stage E.8 · Freeze the full hybrid experiment

Establish the authoritative `RunManifest` using the verified molecular inputs, Stage 2 fragment Hamiltonians, solver assignments, and promoted TrimSQD protocol.

<div class="alert alert-block alert-success">

✏️ **Your task:** Execute the following operations deliberately:

1. Set `RUN_FREEZE_MANIFEST = True` to construct and freeze the scientific manifest.
2. Inspect the resulting solver assignments and verify protocol consistency.
3. Set `RUN_CREATE_FRAGMENT_TASKS = True` to register the fragment calculations.

</div>

The solver allocation is derived from each fragment's verified spatial orbital count:

- **FCI:** Classical fragments at or below `FCI_THRESHOLD`.
- **CCSD:** Remaining classical fragments.
- **TrimSQD:** Fragments selected for quantum treatment in Stage D.1.

**Expected outcome:** `RunManifest: VERIFIED`, with the computed solver distribution and a recorded manifest digest. Task registration reports the number of new and existing fragment tasks.

**Execution is disabled by default.** These operations freeze scientific configuration and register work but do not submit fragment calculations or acquire QPU measurements.

Proceed to **Section 6** to prepare and launch the distributed campaign.

In [ ]:

# ── E.8 · Freeze the RunManifest and register fragment tasks ────────

from collections import Counter
from proteasome_challenge.quantum_coordinator import persist_protocol

RUN_FREEZE_MANIFEST = False
RUN_CREATE_FRAGMENT_TASKS = False

run_manifest = run_ctx.manifest if run_ctx is not None else None

if RUN_FREEZE_MANIFEST:
    if not (
        hf_status.ok
        and ewf_status.ok
        and mf_ref is not None
        and ewf_ref is not None
        and ewf_stage2_result is not None
    ):
        raise RuntimeError(
            "Verified Stage 1 and Stage 2 artifacts are required."
        )

    if (
        mf_ref.run_id != RUN_ID
        or ewf_ref.run_id != RUN_ID
    ):
        raise RuntimeError(
            "Stage artifacts do not belong to the current run."
        )

    if shared_protocol is None:
        raise RuntimeError(
            "Freeze and verify the shared protocol first."
        )

    # Derive the allocation from current-run Stage 2 orbital spaces.
    norb = {
        int(fid): int(n)
        for fid, n in ewf_stage2_result.norb_by_fragment.items()
    }

    catalog_norb = {
        int(row.fragment_id): int(row.n_orb)
        for row in ixa[["fragment_id", "n_orb"]].itertuples(index=False)
    }

    if not norb or norb != catalog_norb:
        raise RuntimeError(
            "Current-run orbital counts differ from the verified catalog."
        )

    quantum_ids = set(map(int, FINAL_FRAGMENT_IDS))

    if (
        not quantum_ids
        or not quantum_ids.issubset(norb)
        or quantum_ids != set(
            map(int, shared_protocol["target_fragment_ids"])
        )
    ):
        raise RuntimeError(
            "Quantum selection differs from the promoted protocol."
        )

    assignments = {
        fid: (
            shared_protocol["solver"]
            if fid in quantum_ids
            else "FCI"
            if n <= FCI_THRESHOLD
            else "CCSD"
        )
        for fid, n in norb.items()
    }

    inputs = collect_manifest_inputs_generic(
        hf_status.record,
        ewf_stage2_result,
        shared_protocol,
        FINAL_FRAGMENT_IDS,
        geometry_path=MOLECULE.path,
        solver_assignments=assignments,
    )

    run_manifest = freeze_run_manifest(
        run_ctx, inputs, mf_ref, ewf_ref
    )

    protocol_path = persist_protocol(run_ctx, shared_protocol)

    if (
        json.loads(protocol_path.read_text())["document_digest"]
        != shared_protocol["document_digest"]
    ):
        raise RuntimeError(
            "Persisted protocol provenance differs."
        )

# Verify the frozen manifest before registering work.
if run_manifest is not None:
    if shared_protocol is None:
        raise RuntimeError("Shared protocol is unavailable.")

    if run_manifest.run_id != RUN_ID:
        raise RuntimeError("RunManifest belongs to a different run.")

    if (
        run_manifest.protocol_digest
        != shared_protocol["protocol_digest"]
    ):
        raise RuntimeError(
            "RunManifest and shared protocol differ."
        )

    solver_assignments = {
        int(fid): solver
        for fid, solver in run_manifest.solver_assignments.items()
    }

    counts = Counter(solver_assignments.values())

    if (
        set(counts) - {"FCI", "CCSD", "TrimSQD"}
        or counts["TrimSQD"] != len(FINAL_FRAGMENT_IDS)
        or {
            fid for fid, solver in solver_assignments.items()
            if solver == "TrimSQD"
        } != set(map(int, FINAL_FRAGMENT_IDS))
    ):
        raise RuntimeError(
            "Hybrid solver allocation differs from the frozen selection."
        )

    print(f"RunManifest: {run_manifest.run_id}")
    print(f"Manifest digest: {run_manifest.manifest_digest}")
    print(
        "Allocation: "
        f"{counts['FCI']} FCI + "
        f"{counts['CCSD']} CCSD + "
        f"{counts['TrimSQD']} TrimSQD"
    )
    print("RunManifest: VERIFIED")

else:
    print("RunManifest: PENDING")

if RUN_CREATE_FRAGMENT_TASKS:
    if run_manifest is None:
        raise RuntimeError(
            "Freeze the RunManifest before registering tasks."
        )

    if ewf_stage2_result is None:
        raise RuntimeError(
            "Verified Stage 2 fragment data is required."
        )

    task_summary = ensure_fragment_tasks(
        run_ctx.registry,
        run_manifest,
        ewf_stage2_result.hamiltonian_hashes,
    )

    print(
        f"Fragment tasks: {len(task_summary['created'])} created, "
        f"{len(task_summary['existing'])} existing"
    )
else:
    print("Fragment task registration: DISABLED")


## 6 · Full quantum measurement campaign

The full measurement campaign extends the single-fragment experiment to every quantum-assigned fragment in the frozen RunManifest. QPU measurements supply occupation-number configurations for SQD and TrimSQD; subsequent classical post-processing produces fragment energies and reduced density matrices required for EWF molecular reconstruction. [Robledo-Moreno et al. (2025)](https://doi.org/10.1126/sciadv.adu9991); [Shajan et al. (2026)](https://doi.org/10.1021/acs.jctc.6c00364).

### Section 6 · Operating the full hybrid campaign

The full hybrid campaign is a **persistent, asynchronous experiment**. Serverless and QPU jobs continue running after notebook execution returns.

The authoritative workload is defined by the frozen `RunManifest`. Every assigned fragment must complete before molecular energy reconstruction.

**01 · Prepare**

Complete **6.1–6.1.2** to establish campaign authorization and verify launch readiness.

Inspect the frozen solver assignments, QPU backend, acquisition settings, and computational resources.

Use the execution controls established in the notebook's control panel:

- `USE_FULL_LIVE_TRIAL` — Selects the full hybrid campaign workflow.
- `ARM_FULL_LIVE_TRIAL` — Authorizes campaign execution.
- `LAUNCH_FULL_TRIAL` — Enables initial workload submission.
- `ADVANCE_FULL_TRIAL` — Enables subsequent reconciliation and advancement.

**All execution authorization flags remain disabled until deliberately enabled.**

**02 · Launch**

After the readiness check passes, explicitly enable `ARM_FULL_LIVE_TRIAL` and `LAUNCH_FULL_TRIAL` in the control panel. Keep `ADVANCE_FULL_TRIAL` disabled.

Run **6.2** to submit the classical and quantum workloads specified by the frozen manifest.

Inspect the launch report for successful, deferred, failed, or ambiguous submissions. Preserve the original `RUN_ID` and recorded provider job identifiers.

**Launch once.** Disable `LAUNCH_FULL_TRIAL` immediately afterward.

**03 · Advance**

Keep `LAUNCH_FULL_TRIAL` disabled. Explicitly enable `ADVANCE_FULL_TRIAL` when ready to reconcile the campaign.

Run **6.3** to perform one bounded reconciliation pass. Each pass may:

- Refresh remote job states.
- Retrieve completed calculations and quantum measurements.
- Dispatch eligible TrimSQD post-processing.
- Update the persistent fragment task registry.

Repeat as necessary without overlapping advancement passes.

A completed QPU job does not necessarily mean its fragment calculation is complete. Quantum measurements must also undergo the required classical post-processing.

Use **6.4–6.4.1** for read-only status inspection.

**04 · Recover failures**

Use **6.5** to diagnose unsuccessful tasks and recover eligible work.

Preserve completed calculations, measurement artifacts, the frozen protocol, and all run-scoped provenance.

Do not restart the entire campaign merely because individual tasks remain unfinished.

A reconciliation pass with no new completions is not necessarily a failure.

**05 · Verify completion**

The campaign is complete only when **every fragment assigned in the frozen `RunManifest` reaches `COMPLETE`**, including quantum-fragment post-processing.

Verify the completion barrier and the integrity of the recorded fragment results.

Proceed to **Section 7** for molecular energy reconstruction.

**QPU execution:** The independent single-fragment laboratory uses `LIVE_QPU_SAMPLING` to select live acquisition or replay. The full campaign has its own authorization and submission workflow. Do not treat `LIVE_QPU_SAMPLING = False` as a safeguard against full-campaign QPU submission.

### Stage 6.1 · Prepare the quantum campaign

Establish run-scoped authorization using the frozen `RunManifest` and promoted TrimSQD protocol.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run **6.1.1** to inspect the quantum fragment assignments, protocol consistency, QPU backend, and measurement settings.

</div>

When satisfied, set `AUTHORIZE_QUANTUM_CAMPAIGN = True` and execute 6.1.1 to create or recover `campaign_auth`.

**Expected outcome:** A valid campaign authorization covering all quantum-assigned fragments, with the remaining QPU allocation reported.

**Authorization is disabled by default.** This stage does not submit QPU jobs.

Proceed to **6.1.2** to verify launch readiness.

In [ ]:

# ── 6.1.1 · Authorize the quantum campaign ───────────────────────

AUTHORIZE_QUANTUM_CAMPAIGN = False

if "campaign_auth" not in globals():
    campaign_auth = None

if not USE_FULL_LIVE_TRIAL or run_ctx is None or run_ctx.manifest is None:
    raise RuntimeError("A frozen FULL_LIVE_TRIAL RunManifest is required.")

run_manifest = run_ctx.manifest

quantum_ids = sorted(
    int(fid)
    for fid, solver in run_manifest.solver_assignments.items()
    if solver in ("SQD", "TrimSQD")
)

if run_manifest.protocol_digest != shared_protocol["protocol_digest"]:
    raise RuntimeError("RunManifest and shared protocol differ.")

if quantum_ids != sorted(map(int, shared_protocol["target_fragment_ids"])):
    raise RuntimeError("Quantum fragment selection mismatch.")

if N_SHOTS != FULL_TRIAL_SHOTS or not QPU_BACKEND:
    raise RuntimeError("Invalid QPU acquisition configuration.")

# Inspect the quantum workload without authorizing execution.
plan_df = pd.DataFrame({
    "fragment_id": quantum_ids,
    "solver": [
        run_manifest.solver_assignments[str(fid)]
        for fid in quantum_ids
    ],
    "backend": QPU_BACKEND,
    "shots": N_SHOTS,
})

print(f"Run: {RUN_ID}")
print(f"Backend: {QPU_BACKEND}")
print(f"QPU fragments: {len(quantum_ids)}")
print(f"LUCJ repetitions: {N_REPS}")

display(plan_df)

if AUTHORIZE_QUANTUM_CAMPAIGN:
    # Create or recover run-scoped campaign authorization.
    campaign_auth = ft.trial_campaign(
        run_ctx,
        run_manifest,
        backend=QPU_BACKEND,
        fragment_ids=quantum_ids,
        shots=N_SHOTS,
        project_root=lab.ckpt_dir.parent,
    )

    print(f"Campaign: {campaign_auth.campaign_id}")
    print(
        "Remaining QPU allocation: "
        f"{campaign_auth.remaining_allocation()}"
    )
else:
    print("Campaign authorization: DISABLED")

# Retained for compatibility with status inspection.
measurement_plan = None


### Stage 6.1.2 · Verify launch readiness

Verify that the frozen scientific configuration, campaign authorization, fragment tasks, and remote execution services are ready.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to check launch readiness for every fragment assigned in the frozen `RunManifest`.

</div>

The preflight verifies:

- **Scientific configuration:** The frozen manifest and shared solver protocol are consistent.
- **Campaign authorization:** The quantum campaign has valid run-scoped authorization.
- **Fragment workload:** Classical and quantum fragment tasks are correctly registered.
- **Execution services:** The required QPU and Serverless interfaces are available.

The preflight reports `PASS` or `BLOCKED`, along with the computed classical and quantum fragment counts, warnings, and launch blockers.

**Expected outcome:** `Launch readiness: PASS`.

This check evaluates readiness assuming launch is armed but **does not submit calculations or acquire QPU measurements**. It may contact remote execution services.

Resolve all blockers before proceeding to **6.2** to launch the full hybrid campaign.

In [ ]:
# ── 6.1.2 · Verify launch readiness ────────────────────────────────

from proteasome_challenge.quantum_coordinator import build_qpu_service_from_env

if campaign_auth is None:
    raise RuntimeError("Initialize campaign authorization in 6.1.1.")

preflight = ft.verify_launch_conditions(
    run_ctx,
    campaign_auth,
    FULL_TRIAL_PROFILE,
    armed=True,  # Test readiness assuming armed; does not launch.
    qpu_service=build_qpu_service_from_env(),
    serverless_service=ft.build_live_interfaces(lab._connect())["service"],
)

print(f"Launch readiness: {'PASS' if preflight['ok'] else 'BLOCKED'}")
print(f"Classical fragments: {len(preflight['classical_fragment_ids'])}")
print(f"Quantum fragments: {len(preflight['quantum_fragment_ids'])}")

for category in ("blockers", "warnings"):
    for message in preflight[category]:
        print(f"{category.upper()}: {message}")

if not preflight["ok"]:
    raise RuntimeError("Resolve launch blockers before proceeding.")

### 6.2 · Launch the full hybrid EWF campaign

Launch the distributed hybrid calculation defined by the frozen `RunManifest`. Classical FCI/CCSD fragment calculations execute on Serverless, while quantum-assigned fragments require QPU measurements followed by classical post-processing.

<div class="alert alert-block alert-success">

✏️ **Your task:** After the **6.1.2** readiness check passes, verify the campaign execution controls:

- `USE_FULL_LIVE_TRIAL = True` — Select the full hybrid workflow.
- `ARM_FULL_LIVE_TRIAL = True` — Authorize campaign execution.
- `LAUNCH_FULL_TRIAL = True` — Enable initial workload submission.
- `ADVANCE_FULL_TRIAL = False` — Keep subsequent reconciliation disabled during launch.

</div>

Set these controls deliberately in the notebook's control panel.

Execute **6.2 once** to submit the classical and quantum workloads specified by the frozen manifest.

**Expected outcome:** A launch report identifying submitted, deferred, or unsuccessful operations, with their associated run-scoped provenance.

After submission, immediately restore `LAUNCH_FULL_TRIAL = False`.

**Do not relaunch the campaign to recover unfinished tasks.** Preserve the existing `RUN_ID`, frozen protocol, recorded job identifiers, and completed work.

If submission status is uncertain, inspect the recorded task states before taking further action.

Proceed to **6.3** to reconcile jobs, retrieve completed results, and advance the remaining work.

In [ ]:

# ── 6.2 · Launch the full hybrid EWF campaign ───────────────────────

ft.assert_legacy_flags_off(globals())

full_trial_launch = None

if run_ctx is None or run_ctx.manifest is None:
    raise RuntimeError("Freeze the RunManifest before launching.")

if LAUNCH_FULL_TRIAL:
    if not USE_FULL_LIVE_TRIAL:
        raise RuntimeError("Enable the full hybrid workflow first.")

    if ADVANCE_FULL_TRIAL:
        raise RuntimeError(
            "Launch and advancement cannot be enabled simultaneously."
        )

    if not ARM_FULL_LIVE_TRIAL:
        raise RuntimeError("Arm the campaign before launching.")

    if campaign_auth is None:
        raise RuntimeError("Complete campaign authorization in 6.1.1.")

    full_trial_launch = ft.notebook_launch(
        lab,
        run_ctx,
        campaign_auth,
        profile=FULL_TRIAL_PROFILE,
        armed=ARM_FULL_LIVE_TRIAL,
    )

    print(ft.format_operation(full_trial_launch))
    print("\nReview the launch report and disable LAUNCH_FULL_TRIAL.")
else:
    print(f"Campaign launch disabled · Run {run_ctx.run_id}")


### 6.3 · Advance the full hybrid campaign

Advance the distributed hybrid experiment by reconciling remote jobs, retrieving completed results, and dispatching eligible TrimSQD post-processing.

<div class="alert alert-block alert-success">

✏️ **Your task:** After launching the campaign in **6.2**, configure the execution controls in the notebook's control panel:

- `USE_FULL_LIVE_TRIAL = True` — Select the full hybrid workflow.
- `ARM_FULL_LIVE_TRIAL = True` — Authorize campaign execution.
- `LAUNCH_FULL_TRIAL = False` — Prevent another initial launch.
- `ADVANCE_FULL_TRIAL = True` — Enable campaign advancement.

</div>

Execute the advancement cell to perform one reconciliation pass.

Each pass updates the persistent task registry and reports campaign progress. **Repeat as needed** until every fragment assigned in the frozen `RunManifest` reaches `COMPLETE`, including quantum-fragment post-processing.

Use **6.4–6.4.1** for read-only monitoring. Do not run overlapping advancement passes or relaunch the campaign.

**Expected outcome:** All assigned FCI, CCSD, and TrimSQD fragment calculations completed and recorded with valid run-scoped provenance.

**Advancement is disabled by default.** Enabling it may dispatch remote classical calculations and retrieve completed jobs.

In [ ]:
# ── 6.3 · Advance the full hybrid EWF campaign ──────────────────────

ft.assert_legacy_flags_off(globals())

if not USE_FULL_LIVE_TRIAL or run_ctx is None or run_ctx.manifest is None:
    raise RuntimeError("A frozen FULL_LIVE_TRIAL RunManifest is required.")

if campaign_auth is None:
    raise RuntimeError("Initialize campaign authorization in 6.1.1.")

full_trial_advance = None

if ADVANCE_FULL_TRIAL:
    if not ARM_FULL_LIVE_TRIAL:
        raise RuntimeError("Arm the campaign before advancement.")

    full_trial_advance = ft.notebook_advance(
        lab,
        run_ctx,
        campaign_auth,
        profile=FULL_TRIAL_PROFILE,
        armed=ARM_FULL_LIVE_TRIAL,
    )
    print(ft.format_operation(full_trial_advance))

print(ft.format_status(ft.full_trial_status(run_ctx, campaign_auth)))

### 6.4 · Inspect campaign status

Inspect the full hybrid campaign without advancing execution.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to review overall task completion, quantum fragment status, and allocated computational resources.

</div>

The cell reports:

- **Campaign status:** Completed, pending, and failed fragment tasks.
- **Quantum execution:** Detailed status of quantum-treated fragments.
- **Resource allocation:** Configured Serverless resources and effective concurrency.

The expected workload is derived from the frozen `RunManifest`, rather than a hard-coded fragment count.

Before the initial launch, the cell also displays the clean-start readiness report.

**Expected outcome:** A campaign status summary identifying completed and outstanding fragment calculations.

This cell is **read-only**, submits no calculations, and can be rerun safely. It may query remote execution services.

Return to **6.3** to advance unfinished work. Proceed to **Section 7** only after every fragment assigned in the frozen manifest has completed successfully.

In [ ]:
# ── 6.4 · Inspect full campaign status ──────────────────────────────

from proteasome_challenge.quantum_coordinator import inspect_quantum_campaign

if run_ctx is None or run_ctx.manifest is None:
    raise RuntimeError("A frozen RunManifest is required.")

if campaign_auth is None:
    raise RuntimeError("Initialize campaign authorization in 6.1.1.")

# Inspect all fragments assigned in the frozen RunManifest.
print(ft.format_status(ft.full_trial_status(run_ctx, campaign_auth)))

# Quantum task breakdown.
print()
print(inspect_quantum_campaign(run_ctx, campaign_auth).summary_table())

# Resource allocation and effective concurrency.
print()
print(ft.describe_profile(FULL_TRIAL_PROFILE, run_ctx))

# Prelaunch acceptance is applicable only before the first launch.
acceptance = ft.clean_slate_report(lab, run_ctx, campaign_auth)
if acceptance["phase"] == "PRE_LAUNCH":
    print()
    print(ft.format_clean_slate_report(acceptance))

### 6.4.1 · Monitor TrimSQD worker progress

Inspect the Serverless execution status and available iteration logs for every TrimSQD fragment assigned in the frozen `RunManifest`.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to examine each worker's status, iteration progress, and latest reported energy.

</div>

The monitoring table displays:

- **Fragment:** The fragment identifier.
- **Provider:** Current Serverless job status.
- **Iteration:** Latest reported TrimSQD iteration.
- **Latest energy (Ha):** Most recent intermediate energy.

Intermediate energies are provisional and should not be interpreted as validated final results.

**Expected outcome:** A status table covering the registered TrimSQD fragment tasks.

This cell is **read-only**, submits no calculations, and can be rerun safely. It may contact Serverless to retrieve job statuses and logs.

Return to **6.3** to reconcile completed work and advance the campaign. Final fragment results must be verified before molecular reconstruction.

In [ ]:
# ── 6.4.1 · Monitor TrimSQD worker progress (read-only) ────────────

import re
import pandas as pd

service = lab._connect()
rows = []

for task in sorted(run_ctx.registry.list_tasks(RUN_ID), key=lambda t: t.fragment_id):
    if task.solver != "TrimSQD":
        continue

    status, iteration, energy = "NOT SUBMITTED", "—", None

    if task.serverless_job_id:
        try:
            job = service.job(task.serverless_job_id)
            status = str(job.status())

            matches = re.findall(
                r"\[TrimSQD iter\s+(\d+)/(\d+)\]\s+E\s*=\s*([-+]?\d+\.\d+)",
                job.logs() or "",
            )

            if matches:
                step, total, value = matches[-1]
                iteration, energy = f"{step}/{total}", float(value)

        except Exception as exc:
            status = f"QUERY ERROR: {exc}"

    rows.append({
        "Fragment": task.fragment_id,
        "Provider": status,
        "Iteration": iteration,
        "Latest energy (Ha)": energy,
    })

display(pd.DataFrame(rows))

### 6.5 · Recover failed fragments

Individual fragment calculations may fail without invalidating the entire hybrid campaign. Recovery should preserve completed work, verified quantum measurements, and the frozen scientific protocol.

<div class="alert alert-block alert-success">

✏️ **Your task:** Use the following cells when a fragment fails:

1. **6.5.1 · Diagnose:** Inspect failed tasks, recorded errors, Serverless job status, and available execution logs.
2. **6.5.2 · Recover:** Requeue eligible failed TrimSQD post-processing using preserved, verified quantum measurements.

</div>

Before recovery, identify the affected fragment and establish the cause of failure. Confirm that its recorded measurement artifact and checksum are valid.

**Recovery must not invalidate completed fragments, alter the frozen `RunManifest`, or discard run-scoped provenance.**

These cells are optional recovery templates, disabled until deliberately activated. Recovery of classical FCI/CCSD failures or failed QPU acquisition may require separate procedures.

After recovery, return to **6.3** to advance the campaign. Molecular reconstruction requires every fragment assigned in the frozen manifest to reach `COMPLETE`.

In [ ]:

# ── 6.5.1 · Diagnose failed fragments (read-only) ────────────────

from proteasome_challenge.task_registry import TaskState

DIAGNOSE_FAILED_FRAGMENTS = False

if DIAGNOSE_FAILED_FRAGMENTS:
    if run_ctx is None:
        raise RuntimeError("Initialize the run context first.")

    failed = [
        t for t in run_ctx.registry.list_tasks(RUN_ID)
        if t.state == TaskState.FAILED
    ]

    if not failed:
        print("No failed fragments.")
    else:
        service = lab._connect()

        for task in failed:
            print(f"\nFragment {task.fragment_id}")
            print("Reason:", task.failure_reason)
            print("QPU job:", task.qpu_job_id)
            print("Serverless job:", task.serverless_job_id)
            print(
                "Post-processing attempt:",
                task.extra.get("pp_attempt", 0),
            )

            if task.serverless_job_id:
                try:
                    job = service.job(task.serverless_job_id)
                    print("Provider status:", job.status())
                    print("Logs:\n", (job.logs() or "")[-4000:])

                    try:
                        print(
                            "Provider result:",
                            str(job.result())[:2000],
                        )
                    except Exception as exc:
                        print(
                            "Provider error:",
                            type(exc).__name__,
                            str(exc)[:2000],
                        )

                except Exception as exc:
                    print(
                        "Inspection error:",
                        type(exc).__name__,
                        str(exc),
                    )
else:
    print("Failed-fragment diagnosis: DISABLED")


In [ ]:

# ── 6.5.2 · Recover failed TrimSQD post-processing ───────────────

import hashlib
from pathlib import Path

from proteasome_challenge.task_registry import TaskState
from proteasome_challenge.quantum_coordinator import (
    requeue_failed_postprocessing,
)

RUN_MEND_FRAGMENT = False
MEND_FRAGMENT_ID = None  # Set to the failed fragment ID from 6.5.1.

if RUN_MEND_FRAGMENT:
    if run_ctx is None or run_ctx.run_id != RUN_ID:
        raise RuntimeError("A valid current-run context is required.")

    if MEND_FRAGMENT_ID is None:
        raise ValueError(
            "Select the failed TrimSQD fragment identified in 6.5.1."
        )

    failed = [
        task
        for task in run_ctx.registry.list_tasks(RUN_ID)
        if task.solver == "TrimSQD"
        and task.state == TaskState.FAILED
    ]

    # The recovery API is not scoped to an individual fragment.
    # Require exactly one failed TrimSQD task before proceeding.
    if len(failed) != 1 or failed[0].fragment_id != MEND_FRAGMENT_ID:
        raise RuntimeError(
            "Recovery requires exactly one failed TrimSQD fragment, "
            "matching MEND_FRAGMENT_ID. Review all failures first."
        )

    task = failed[0]

    counts_path = task.extra.get("pp_counts_path")
    expected_sha = task.extra.get("pp_counts_sha256")

    if not counts_path or not expected_sha or not task.qpu_job_id:
        raise RuntimeError(
            "Preserved QPU measurements or provenance are unavailable."
        )

    counts = Path(counts_path).resolve()
    run_dir = Path(run_ctx.paths.run_dir).resolve()

    if not counts.is_relative_to(run_dir):
        raise RuntimeError(
            "Measurement file is outside the current run."
        )

    if not counts.is_file():
        raise RuntimeError("Preserved measurement file is missing.")

    with counts.open("rb") as f:
        actual_sha = hashlib.file_digest(f, "sha256").hexdigest()

    if actual_sha != expected_sha:
        raise RuntimeError(
            "Preserved measurement checksum mismatch."
        )

    old_qpu_job = task.qpu_job_id

    result = requeue_failed_postprocessing(
        run_ctx,
        max_attempts=3,
        reason="Operator retry after failed TrimSQD post-processing",
    )

    if result["requeued"] != [MEND_FRAGMENT_ID]:
        raise RuntimeError(
            f"Unexpected recovery result: {result}"
        )

    requeued = run_ctx.registry.get_task(task.task_id)

    if not (
        requeued.state == TaskState.SAMPLES_READY
        and requeued.qpu_job_id == old_qpu_job
        and not requeued.serverless_job_id
    ):
        raise RuntimeError(
            "Recovered task failed post-recovery verification."
        )

    print(
        f"Fragment {MEND_FRAGMENT_ID}: READY FOR POST-PROCESSING"
    )
    print(f"Original QPU job preserved: {old_qpu_job}")
    print("Return to 6.3 to dispatch the replacement worker.")

else:
    print("TrimSQD recovery: DISABLED")


In [ ]:

# Exercise 5 — Submit TrimSQD fragment energies

import math

from proteasome_challenge.task_registry import TaskState
from serverless_bundle.steps.fragment_result_store import load_fragment_result

SUBMIT_EXERCISE_5 = False

if SUBMIT_EXERCISE_5:
    if (
        run_ctx is None
        or run_ctx.manifest is None
        or run_ctx.run_id != RUN_ID
    ):
        raise RuntimeError(
            "A frozen RunManifest for the current run is required."
        )

    # Derive the expected quantum fragments from the frozen manifest.
    expected_ids = {
        int(fid)
        for fid, solver in run_ctx.manifest.solver_assignments.items()
        if solver == "TrimSQD"
    }

    tasks = [
        task
        for task in run_ctx.registry.list_tasks(RUN_ID)
        if task.solver == "TrimSQD"
    ]

    actual_ids = [int(task.fragment_id) for task in tasks]

    if (
        not expected_ids
        or len(actual_ids) != len(set(actual_ids))
        or set(actual_ids) != expected_ids
        or any(task.state != TaskState.COMPLETE for task in tasks)
    ):
        raise RuntimeError(
            "TrimSQD fragment tasks are incomplete or differ "
            "from the frozen RunManifest."
        )

    bundle = {}

    for task in sorted(tasks, key=lambda t: t.fragment_id):
        result = load_fragment_result(
            task.result_artifact_uri,
            expected_fragment_id=task.fragment_id,
            expected_run_id=RUN_ID,
            expected_solver=task.solver,
            expected_hamiltonian_sha256=task.hamiltonian_sha256,
        )

        energy = float(result.energy)

        if not math.isfinite(energy):
            raise RuntimeError(
                f"Non-finite energy for fragment {task.fragment_id}."
            )

        bundle[f"fragment_{task.fragment_id}_energy"] = energy

    print(f"Verified {len(bundle)} TrimSQD fragment energies.")

    grade_lab_proteasome_ex5(bundle)

else:
    print("Exercise 5 submission: DISABLED")


## 7 · Reconstruct the molecular electronic energy

The embedded fragment calculations produce correlated wavefunctions within overlapping **fragment-and-bath orbital spaces**. Because these spaces overlap, their cluster energies cannot simply be added to obtain the molecular energy.

Instead, **partitioned-cumulant reconstruction** combines information from each fragment's reduced density matrices (RDMs):

- **One-particle RDM (1-RDM):** Describes electronic occupations and coherence between orbitals. Changes relative to the Hartree–Fock reference contribute to the correlation energy.
- **Two-particle RDM (2-RDM):** Describes electron-pair properties. Its cumulant isolates correlation information beyond contributions constructed from one-particle densities.
- **Fragment projector:** Assigns contributions from the embedded cluster to its corresponding fragment orbitals, accounting for overlapping fragment-and-bath spaces.

The reconstructed molecular energy is assembled as

$$
E_{\mathrm{EWF}}
=
E_{\mathrm{HF}}
+
\sum_{x=1}^{N_{\mathrm{frag}}}
\left(
E_{1,x}^{\mathrm{PC}}
+
E_{2,x}^{\mathrm{PC}}
\right).
$$

Here, $N_{\mathrm{frag}}$ is the number of fragments in the verified Stage 2 decomposition, $E_{\mathrm{HF}}$ is the Hartree–Fock reference energy, and $E_{1,x}^{\mathrm{PC}}$ and $E_{2,x}^{\mathrm{PC}}$ represent the one- and two-body partitioned-cumulant contributions from fragment $x$.

All FCI, CCSD, and TrimSQD fragments assigned in the frozen `RunManifest` enter through the same reconstruction procedure, using the reduced density matrices supplied by their respective solvers.

**References:** [Nusspickel and Booth (2022)](https://doi.org/10.1103/PhysRevX.12.011046) · [Nusspickel, Ibrahim, and Booth (2023)](https://doi.org/10.1021/acs.jctc.2c01063)

### Exercise 6 · Reconstruct the molecular energy

**Objective:** Assemble the correlated fragment contributions into a molecular electronic energy, verify the reconstruction, and interpret the contributions from the different solver methods.

**01 · Verify the completed calculations**

**F.0 · Reconstruction preflight**

Before reconstruction, run the preflight verification cell.

Confirm that:

- Every fragment assigned in the frozen `RunManifest` has completed.
- The FCI, CCSD, and TrimSQD assignments match the frozen solver allocation.
- The fragment results agree with the frozen `RunManifest`.
- The required quantum measurement provenance is valid.
- The scientific completion barrier passes.

Reconstruction requires the completed fragment results, including their reduced density matrices—not merely the fragment energies.

**02 · Reconstruct the molecular energy**

**F.1 · Molecular EWF energy reconstruction**

The reconstruction is performed as a separate Serverless calculation.

Use the following execution controls:

| Control | Purpose |
|---|---|
| `STAGE4_ACTION = "dispatch"` | Submit a new reconstruction job |
| `STAGE4_ACTION = "reconcile"` | Retrieve and register the result of an existing job |
| `STAGE4_ACTION = "inspect"` | Inspect recorded status without submitting or reconciling |
| `STAGE4_RESOURCE_FAMILY = "l"` | Select the Serverless resource family for dispatch |

The default action is `"inspect"`, which does not submit calculations.

For a new reconstruction, deliberately set `STAGE4_ACTION = "dispatch"` and execute F.1. Dispatch requires the scientific completion barrier to pass and refuses duplicate or uncertain submissions.

While reconstruction is running, use **F.1a** to inspect the Serverless worker status and logs.

After the job completes, return to F.1, set `STAGE4_ACTION = "reconcile"`, and execute the cell again to retrieve the authoritative molecular energy.

**Only one action is selected at a time.** Reconstruction does not perform additional QPU sampling.

Once reconciliation completes, inspect the reported molecular energies:

- `e_hf` — Hartree–Fock reference energy
- `e_corr_total` — Reconstructed correlation energy
- `e_total` — Final molecular energy

**03 · Analyze the reconstruction**

**F.3 · Validation and energy accounting**

Run the validation and fragment-accounting cell to inspect how the molecular energy was assembled.

Pay particular attention to:

| Quantity | Interpretation |
|---|---|
| `EWF e_corr (Ha)` | Reconstructed correlation contribution from each fragment |
| `Solver` | FCI, CCSD, or TrimSQD assignment |
| `RDM integrated` | Whether the fragment's RDMs entered the reconstruction |
| `e_corr_total` | Sum of the reconstructed fragment contributions |
| `e_total` | Final molecular energy |

**Questions to investigate**

- How large is the total correlation correction relative to the Hartree–Fock energy?
- How much correlation energy is contributed by the FCI, CCSD, and TrimSQD groups?
- Do all fragment contributions reconcile with the reported correlation energy?
- Does the final energy satisfy the reconstruction identity?

The validation cell checks numerical consistency between the fragment contributions, total correlation energy, and molecular energy. It also verifies the recorded reconstruction job provenance.

**04 · Submit the reconstructed energy**

Run the Exercise 6 submission cell, which extracts the final molecular energy from `stage4_summary["e_total"]` and submits it through `grade_lab_proteasome_ex6(bundle)`.

Grading submission is disabled by default. Enable `SUBMIT_EXERCISE_6` only after verifying the completed reconstruction.

**Grading · Exercise 6**

| Criterion | Requirement |
|---|---|
| Molecular system | Ixazomib |
| Submitted quantity | Reconstructed `e_total` |
| Energy agreement | ±0.01 Ha |
| Result | Correct molecular energy |

**If grading fails:** Verify the fragment energies used in Stage 4 reconstruction. Inspect the reconstruction status and energy accounting, correct any discrepancies, reconcile or rerun reconstruction as appropriate, and resubmit.

A successful submission completes the Challenge: **from molecular specification and Hartree–Fock through EWF fragmentation, quantum-assisted correlated solvers, and reconstruction of the molecular electronic energy.**

### F.0 · Verify readiness for molecular reconstruction

Verify that the completed hybrid campaign satisfies the scientific requirements for molecular energy reconstruction.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to validate the frozen `RunManifest`, solver assignments, fragment completion, and quantum measurement provenance.

</div>

The preflight requires:

- **Complete workload:** Every fragment assigned in the frozen `RunManifest` has completed successfully.
- **Consistent solver allocation:** The completed FCI, CCSD, and TrimSQD assignments match the frozen manifest.
- **Valid quantum provenance:** Every quantum-assigned fragment has verified, non-replayed measurements.
- **Consistent scientific records:** Fragment identities, solver assignments, and results satisfy the strict reconstruction barrier.
- **Unambiguous execution state:** No uncertain reconstruction submissions.

The expected fragment counts are derived from the frozen manifest rather than supplied as reference answers.

**Expected outcome:** `Stage F preflight: VERIFIED`, followed by the computed fragment counts, solver distribution, quantum provenance summary, and manifest digest.

This cell is **read-only** and submits no calculations or QPU jobs.

Once verification passes, proceed to **F.1** for molecular energy reconstruction.

In [ ]:

# ── F.0 · Verify readiness for molecular reconstruction ───────────

from proteasome_challenge import stage4_reconstruction as s4

stage4_ctx = run_ctx
stage4_step = {}

if (
    stage4_ctx is None
    or stage4_ctx.manifest is None
    or stage4_ctx.run_id != RUN_ID
):
    raise RuntimeError("A frozen RunManifest for the current run is required.")

# Verify the completed scientific workload.
barrier_report = s4.run_strict_barrier(stage4_ctx).as_dict()
plan = s4.manifest_composition(stage4_ctx.manifest)

assignments = {
    int(fid): solver
    for fid, solver in stage4_ctx.manifest.solver_assignments.items()
}

expected_solvers = {"FCI", "CCSD", "TrimSQD"}

if (
    not assignments
    or set(assignments.values()) != expected_solvers
):
    raise RuntimeError("Invalid hybrid solver allocation.")

expected_counts = {
    solver: sum(method == solver for method in assignments.values())
    for solver in ("FCI", "CCSD", "TrimSQD")
}

planned_ids = [
    int(fid)
    for ids in plan["fragments_by_solver"].values()
    for fid in ids
]

quantum_ids = {
    fid for fid, solver in assignments.items()
    if solver == "TrimSQD"
}

quantum = barrier_report["quantum_provenance"]
total = len(assignments)

if not (
    barrier_report["passed"]
    and barrier_report["completed"] == total
    and plan["total"] == total
    and len(planned_ids) == len(set(planned_ids)) == total
    and set(planned_ids) == set(assignments)
    and set(planned_ids) == {
        int(fid) for fid in barrier_report["required_fragments"]
    }
    and plan["counts"] == expected_counts
    and barrier_report["actual_composition"] == expected_counts
    and {
        int(fid) for fid in plan["quantum_fragment_ids"]
    } == quantum_ids
    and quantum_ids == set(map(int, FINAL_FRAGMENT_IDS))
    and len(quantum) == len(quantum_ids)
    and all(not q["is_replay"] for q in quantum.values())
):
    raise RuntimeError(
        f"Reconstruction preflight failed: {barrier_report['errors']}"
    )

# Inspect the authoritative reconstruction ledger.
stage4_status = s4.stage4_status(stage4_ctx)
stage4_summary = s4.load_authoritative_summary(stage4_ctx)

if stage4_status["uncertain"]:
    raise RuntimeError(
        "Uncertain Stage 4 submission requires investigation."
    )

print("Stage F preflight: VERIFIED")
print(f"Run: {RUN_ID}")
print(f"Fragments: {total}/{total} COMPLETE")
print(
    "Allocation: "
    f"{expected_counts['FCI']} FCI + "
    f"{expected_counts['CCSD']} CCSD + "
    f"{expected_counts['TrimSQD']} TrimSQD"
)
print(
    f"Quantum provenance: "
    f"{len(quantum)}/{len(quantum_ids)} verified"
)
print(f"Manifest digest: {barrier_report['manifest_digest']}")
print(f"Existing reconstruction jobs: {len(stage4_status['jobs'])}")


### F.1 · Reconstruct the molecular energy

Reconstruct the molecular electronic energy from the completed EWF fragment calculations using a separate Serverless job.

<div class="alert alert-block alert-success">

✏️ **Your task:** Select `STAGE4_ACTION` and execute this cell:

- `"inspect"` — Inspect the recorded reconstruction status without submitting or reconciling.
- `"dispatch"` — Submit a new reconstruction job after the scientific completion barrier passes.
- `"reconcile"` — Retrieve and register the results of an existing reconstruction job.

</div>

**Execution defaults to `"inspect"`.** Dispatch and reconciliation must be selected deliberately.

For a new reconstruction, set `STAGE4_ACTION = "dispatch"` and run this cell once. The workflow refuses duplicate or ambiguous reconstruction submissions.

Use **F.1a** to monitor the Serverless worker. Once the job completes, return here with `STAGE4_ACTION = "reconcile"`.

**Expected outcome:** A verified `stage4_summary` containing:

- `e_hf` — Hartree–Fock reference energy.
- `e_corr_total` — Reconstructed molecular correlation energy.
- `e_total` — Total molecular electronic energy.

Reconstruction requires the completed fragment results, including their reduced density matrices, and performs **no additional QPU sampling**.

Proceed to **F.3** to validate the energy reconstruction and examine fragment contributions.

In [ ]:

# ── F.1 · Molecular EWF energy reconstruction ──────────────────────

from proteasome_challenge import stage4_reconstruction as s4

STAGE4_ACTION = "inspect"  # "inspect", "dispatch", "reconcile"
STAGE4_RESOURCE_FAMILY = "l"

stage4_ctx = run_ctx
stage4_step = None

if stage4_ctx is None or stage4_ctx.manifest is None:
    raise RuntimeError("A frozen RunManifest is required.")
if stage4_ctx.run_id != RUN_ID:
    raise RuntimeError("Run identity mismatch.")
if STAGE4_ACTION not in {"dispatch", "reconcile", "inspect"}:
    raise ValueError("Invalid STAGE4_ACTION.")

status = s4.stage4_status(stage4_ctx)

if STAGE4_ACTION == "dispatch":
    # Never submit a duplicate or ambiguous reconstruction.
    if (
        status["jobs"]
        or status["uncertain"]
        or s4.load_authoritative_summary(stage4_ctx) is not None
    ):
        raise RuntimeError(
            "A reconstruction already exists or is uncertain. "
            "Inspect or reconcile before attempting another submission."
        )

    # One scientific completion barrier, immediately before dispatch.
    barrier = s4.run_strict_barrier(stage4_ctx).as_dict()
    if not barrier["passed"]:
        raise RuntimeError(f"Reconstruction barrier failed: {barrier['errors']}")

    lab._assert_bundle_deployed()

    try:
        stage4_step = s4.run_notebook_step(
            stage4_ctx,
            client=lab._connect(),
            dispatch=True,
            authorize_remote=True,
            function_size=STAGE4_RESOURCE_FAMILY,
        )
    except Exception:
        print("Dispatch failed or is uncertain. Inspect the ledger before retrying.")
        print(s4.stage4_status(stage4_ctx))
        raise

elif STAGE4_ACTION == "reconcile":
    # Retrieve the existing job; no new submission.
    stage4_step = s4.run_notebook_step(
        stage4_ctx,
        client=lab._connect(),
        reconcile=True,
        dispatch=False,
        authorize_remote=False,
    )

# Read-only inspection when STAGE4_ACTION == "inspect".
stage4_status = s4.stage4_status(stage4_ctx)
stage4_summary = s4.load_authoritative_summary(stage4_ctx)

if stage4_step is not None:
    print("Operation:", stage4_step)

print(f"Run: {stage4_ctx.run_id}")
print(f"Jobs: {len(stage4_status['jobs'])}")
print(f"Active: {len(stage4_status['active'])}")
print(f"Uncertain: {len(stage4_status['uncertain'])}")

if stage4_summary is None:
    print("Molecular reconstruction: PENDING")
else:
    print("Molecular reconstruction: COMPLETE")
    print(f"E_HF    = {stage4_summary['e_hf']:.12f} Ha")
    print(f"E_corr  = {stage4_summary['e_corr_total']:.12f} Ha")
    print(f"E_total = {stage4_summary['e_total']:.12f} Ha")


### F.1a · Monitor reconstruction progress

Inspect the Stage 4 Serverless worker's status and execution logs.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell to monitor reconstruction progress or diagnose worker failures.

</div>

The cell retrieves the latest job from the reconstruction ledger. It is **read-only**, performs no calculations, and can be rerun safely.

Once the worker completes, return to **F.1** with `STAGE4_ACTION = "reconcile"` to retrieve the authoritative molecular energy.

In [ ]:
# ── F.1a · Inspect reconstruction worker logs ──────────────────────
# READ ONLY: no submissions or reconciliation.

from proteasome_challenge import stage4_reconstruction as s4

jobs = s4.stage4_status(stage4_ctx)["jobs"]
if not jobs:
    raise RuntimeError("No Stage 4 reconstruction job exists.")

latest = max(jobs, key=lambda j: int(j["attempt"]))
job_id = latest["serverless_job_id"]

if not job_id:
    raise RuntimeError("No Serverless job ID recorded.")

job = lab._connect().job(job_id)

print(f"Run:      {stage4_ctx.run_id}")
print(f"Attempt:  {latest['attempt']}")
print(f"Job:      {job_id}")
print(f"Provider: {job.status()}")

print("\n=== RECENT WORKER LOGS ===")
try:
    logs = str(job.logs() or "")
    print("\n".join(logs.splitlines()[-60:])[-8000:]
          if logs.strip() else "No logs available yet.")
except Exception as exc:
    print(f"Logs unavailable: {type(exc).__name__}: {exc}")

### F.3 · Validate and analyze molecular energy

Verify the completed molecular energy reconstruction against the authoritative Serverless results and frozen `RunManifest`.

<div class="alert alert-block alert-success">

✏️ **Your task:** Run this cell after completing reconciliation in **F.1**.

</div>

The validation checks:

- **Provenance:** The reconstruction originates from a completed Serverless job recorded in the current run.
- **Completeness:** Every fragment assigned in the frozen `RunManifest` appears exactly once.
- **Solver consistency:** Each fragment's recorded solver matches its frozen assignment.
- **RDM integration:** Every fragment's reduced density matrices contributed to the reconstruction.
- **Energy accounting:** Fragment correlation contributions sum to the reported molecular correlation energy.
- **Energy identity:** The reconstructed total energy equals the Hartree–Fock energy plus the correlation correction.

**Expected outcome:** `STAGE F · RECONSTRUCTION VERIFIED`, followed by the Hartree–Fock, correlation, and total molecular energies, contributions grouped by solver, and individual fragment results.

All expected fragment counts are derived from the frozen manifest.

This cell is **read-only** and performs no remote calculations or QPU sampling.

Once the reconstruction is verified, proceed to **Exercise 6** to submit the molecular total energy.

In [ ]:

# ── F.3 · Validate and analyze molecular energy ────────────────────
# READ ONLY: no remote execution.

import math
import pandas as pd
from IPython.display import display
from proteasome_challenge import stage4_reconstruction as s4

if (
    stage4_ctx is None
    or stage4_ctx.manifest is None
    or stage4_ctx.run_id != RUN_ID
):
    raise RuntimeError("A frozen current-run RunManifest is required.")

stage4_summary = s4.load_authoritative_summary(stage4_ctx)

if stage4_summary is None:
    raise RuntimeError(
        "Reconcile the completed reconstruction in F.1 first."
    )

s4.validate_summary(stage4_summary)

# Verify reconstruction against the authoritative job ledger.
execution = stage4_summary["execution"]
job_id = execution["serverless_job_id"]
status = s4.stage4_status(stage4_ctx)

if (
    execution["mode"] != "serverless"
    or status["uncertain"]
    or not any(
        j["serverless_job_id"] == job_id
        and j["state"] == "COMPLETE"
        for j in status["jobs"]
    )
):
    raise RuntimeError("Reconstruction job provenance mismatch.")

# Derive the expected fragment assignments from the frozen manifest.
expected = {
    int(fid): solver
    for fid, solver in stage4_ctx.manifest.solver_assignments.items()
}

# Verify exact fragment coverage before constructing the analysis table.
fragment_records = stage4_summary["fragments"]
actual = {
    int(fid): rec
    for fid, rec in fragment_records.items()
}

if (
    not expected
    or len(actual) != len(fragment_records)
    or set(actual) != set(expected)
    or any(
        rec["solver"] != expected[fid]
        for fid, rec in actual.items()
    )
):
    raise RuntimeError(
        "Reconstructed fragments differ from the frozen RunManifest."
    )

fragment_df = pd.DataFrame([
    {
        "Fragment": fid,
        "Solver": rec["solver"],
        "Orbitals": rec["n_orb"],
        "EWF e_corr (Ha)": float(rec["e_corr"]),
        "RDM integrated": bool(rec["rdm_integrated"]),
    }
    for fid, rec in actual.items()
]).sort_values("Fragment")

e_hf = float(stage4_summary["e_hf"])
e_corr = float(stage4_summary["e_corr_total"])
e_total = float(stage4_summary["e_total"])

fragment_energies = fragment_df["EWF e_corr (Ha)"].tolist()

if not all(math.isfinite(value) for value in (
    e_hf, e_corr, e_total, *fragment_energies
)):
    raise RuntimeError("Non-finite reconstruction energy.")

residual = math.fsum(fragment_energies) - e_corr

if (
    not fragment_df["RDM integrated"].all()
    or abs(residual) >= 1e-9
    or abs(e_total - math.fsum((e_hf, e_corr))) >= 1e-9
):
    raise RuntimeError("Reconstruction energy accounting failed.")

print("STAGE F · RECONSTRUCTION VERIFIED")
print(f"Fragments: {len(fragment_df)}/{len(expected)}")
print(f"E_HF:      {e_hf:.12f} Ha")
print(f"E_corr:    {e_corr:.12f} Ha")
print(f"E_total:   {e_total:.12f} Ha")
print(f"Residual:  {residual:.3e} Ha")

print("\nCorrelation energy by solver:")
for solver, group in fragment_df.groupby("Solver"):
    print(
        f"{solver:8s}  {len(group):2d} fragments  "
        f"{math.fsum(group['EWF e_corr (Ha)']):+.10f} Ha"
    )

print("\nFragment contributions:")
display(fragment_df)


In [ ]:

# Exercise 6 — Ixazomib Total Energy

import math

SUBMIT_EXERCISE_6 = False

if SUBMIT_EXERCISE_6:
    if (
        "stage4_summary" not in globals()
        or stage4_summary is None
    ):
        raise RuntimeError(
            "Complete and verify molecular reconstruction in F.3 "
            "before submitting Exercise 6."
        )

    energy = float(stage4_summary["e_total"])

    if not math.isfinite(energy):
        raise RuntimeError("Invalid molecular total energy.")

    bundle = {
        "ixazomib_total_energy": energy,
    }

    grade_lab_proteasome_ex6(bundle)

else:
    print("Exercise 6 submission: DISABLED")


### Final Experiment Report · Grader Verification

This report provides the **authoritative experimental record** for verifying challenge submissions and grader results.

It consolidates run-scoped configurations, computed results, execution outcomes, failures, and provenance, distinguishing completed, failed, pending, and unexecuted experiments.

**Winner verification:** For any potentially winning submission, organizers will use this report to independently verify the graded results and determine final eligibility, including confirmation of a win or disqualification if the submitted results cannot be substantiated.

The following cells generate the report in HTML, JSON, and CSV formats for organizer review.

In [ ]:
# ── O.1 · Final experiment audit ─────────────────────────
# Read-only scientific audit; writes local HTML, JSON, and CSV reports.

from pathlib import Path
from IPython.display import HTML, display
from proteasome_challenge.experiment_report import build_experiment_report

# 1. Build the report from the current experiment.
report = build_experiment_report(
    run_ctx,
    fragment_lab=globals().get("fragment_lab"),
    active_fragment_id=globals().get("ACTIVE_FRAGMENT_ID"),
)

# 2. Export directly into the current run's checkpoint directory.
report_dir = (
    Path(run_ctx.paths.run_dir).resolve() / "organizer_report"
)

files = report.export(report_dir)

# 3. Verify every exported file exists and is nonempty.
for name, path in files.items():
    path = Path(path).resolve()
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"Report export failed: {name} → {path}")

# 4. Confirm the exact save location.
campaign = report.data["campaign"]

print("=== ORGANIZER EXPERIMENT AUDIT ===")
print(f"Run:       {run_ctx.run_id}")
print(f"Fragments: {campaign['tasks_complete']}/{campaign['tasks_planned']}")
print()
print("REPORT SUCCESSFULLY SAVED")
print(f"Directory: {report_dir}")
print()

for name, path in files.items():
    path = Path(path).resolve()
    print(f"✓ {path.name} ({path.stat().st_size:,} bytes)")

print()
print(f"OPEN IN VS CODE: {report_dir / 'experiment_report.html'}")

In [ ]:
# ── Render report with isolated, readable light styling ────────────

from html import escape

page = report.to_html()

# Explicit light theme for the exported HTML.
page = page.replace(
    "<style>",
    "<style>html, body { "
    "background: #ffffff !important; "
    "color: #20242b !important; "
    "color-scheme: light !important; "
    "}",
    1,
)

# Update the saved HTML report as well.
html_path = Path(files["html"]).resolve()
html_path.write_text(page, encoding="utf-8")

# Isolate the report from VS Code's dark notebook theme.
display(HTML(
    f'<iframe title="Organizer Experiment Audit" '
    f'srcdoc="{escape(page, quote=True)}" '
    'style="width:100%;height:950px;'
    'border:1px solid #cbd5e1;'
    'border-radius:6px;background:white;">'
    '</iframe>'
))

print(f"Readable HTML report saved: {html_path}")